In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"

S1_PATH = RAW_DATA_DIR / "train_source1.tsv"
S2_PATH = RAW_DATA_DIR / "train_source2.tsv"
S3_PATH = RAW_DATA_DIR / "train_source3.tsv"
GT_PATH = RAW_DATA_DIR / "train_ground_truth.tsv"

TRAIN_IDS_PATH = (
    INTERIM_DATA_DIR
    / "splits"
    / "train_s1_ids.tsv"
)

VALID_IDS_PATH = (
    INTERIM_DATA_DIR
    / "splits"
    / "valid_s1_ids.tsv"
)

print("Project root:", PROJECT_ROOT)
print("S1:", S1_PATH.exists())
print("S2:", S2_PATH.exists())
print("S3:", S3_PATH.exists())
print("Ground truth:", GT_PATH.exists())
print("Train IDs:", TRAIN_IDS_PATH.exists())
print("Validation IDs:", VALID_IDS_PATH.exists())

Project root: d:\amzn_ml_challenge
S1: True
S2: True
S3: True
Ground truth: True
Train IDs: True
Validation IDs: True


In [2]:
SOURCE_COLUMNS = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

s1 = pd.read_csv(
    S1_PATH,
    sep="\t",
    usecols=SOURCE_COLUMNS,
)

s2 = pd.read_csv(
    S2_PATH,
    sep="\t",
    usecols=SOURCE_COLUMNS,
)

s3 = pd.read_csv(
    S3_PATH,
    sep="\t",
    usecols=SOURCE_COLUMNS,
)

print("S1 shape:", s1.shape)
print("S2 shape:", s2.shape)
print("S3 shape:", s3.shape)

S1 shape: (2206821, 4)
S2 shape: (5034616, 4)
S3 shape: (5285603, 4)


In [3]:
from IPython.display import display
memory_usage = pd.DataFrame({
    "source": ["S1", "S2", "S3"],
    "rows": [len(s1), len(s2), len(s3)],
    "memory_mb": [
        s1.memory_usage(deep=True).sum() / 1024**2,
        s2.memory_usage(deep=True).sum() / 1024**2,
        s3.memory_usage(deep=True).sum() / 1024**2,
    ]
})

display(memory_usage)

print(
    f"\nTotal DataFrame memory: "
    f"{memory_usage['memory_mb'].sum():.2f} MB"
)

,source,rows,memory_mb
0,S1,2206821,604.425661
1,S2,5034616,1418.800255
2,S3,5285603,1478.472595



Total DataFrame memory: 3501.70 MB


In [4]:
for name, df in [("S1", s1), ("S2", s2), ("S3", s3)]:
    print(f"\n{name}")
    print(
        (df.memory_usage(deep=True) / 1024**2)
        .sort_values(ascending=False)
        .round(2)
    )


S1
business_address    212.71
business_name       153.71
entity_id           128.15
country             109.86
Index                 0.00
dtype: float64

S2
business_address    493.90
business_name       381.91
entity_id           292.35
country             250.64
Index                 0.00
dtype: float64

S3
business_address    516.68
business_name       391.74
entity_id           306.92
country             263.13
Index                 0.00
dtype: float64


In [5]:
train_ids = pd.read_csv(
    TRAIN_IDS_PATH,
    sep="\t"
)

gt = pd.read_csv(
    GT_PATH,
    sep="\t"
)

gt["match_count"] = (
    gt["matched_entity_ids"]
    .fillna("")
    .apply(lambda x: 0 if x == "" else len(x.split(",")))
)

train_gt = train_ids.merge(
    gt,
    on="source1_entity_id",
    how="left"
)

print("Training S1 rows:", len(train_gt))
print("\nMatch-count distribution:")
print(
    train_gt["match_count"]
    .value_counts()
    .sort_index()
)

Training S1 rows: 1765456

Match-count distribution:
match_count
0      98598
1      95326
2     300169
3     424673
4     387292
5     257565
6     131894
7      51174
8      14944
9       3364
10       427
11        30
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

BLOCKING_SAMPLE_SIZE = 10_000

blocking_sample, _ = train_test_split(
    train_gt,
    train_size=BLOCKING_SAMPLE_SIZE,
    random_state=42,
    stratify=train_gt["match_count"],
)

print("Blocking sample size:", len(blocking_sample))

print("\nMatch-count distribution:")
print(
    blocking_sample["match_count"]
    .value_counts()
    .sort_index()
)

Blocking sample size: 10000

Match-count distribution:
match_count
0      559
1      540
2     1700
3     2405
4     2194
5     1459
6      747
7      290
8       85
9       19
10       2
Name: count, dtype: int64


In [7]:
display(
    s1[
        ["entity_id", "business_name", "business_address", "country"]
    ].sample(20, random_state=42)
)

,entity_id,business_name,business_address,country
2195840,S1-53356671,Pediatric Medicine PLLC,"4850 20, Otisco, NY",US
1487051,S1-320151505,Fetech National Twin,"19034 Woodburn Road, Woodburn, IN",US
1878277,S1-938947364,General Design Innovations LLC,"7241 Osage Avenue, Mesa, AZ",US
993966,S1-195839862,Construction Ideaz Papers Private Limited,"Building No.4/606, Prabhul Cottage Karimbalur,...",India
565428,S1-655046555,Penaloza and Bittle First Inc.,"4255 Charleswood Avenue, Memphis, TN",US
1645753,S1-758070915,National Investments LLC,"1641 Virginia Lane, Hueytown, AL",US
1165724,S1-169338169,Reus Idaho Company,"Sanford, ME, 31 Guillemette Street",US
363842,S1-796421498,Creative Projects Limited,"12Thfloor, C, 1204, Roya Oasis, Jankalyan Naga...",India
962445,S1-802535179,Goar Stone LLC,"2105 Wood Ridge Cove, Cedar Park, TX",US
964820,S1-97176033,Lakshmi Consultants Private Limited,"A-301, New Sai Dham Chsl, Ramdev Park Road, Th...",India


In [8]:
non_ascii_s1 = s1[
    s1["business_name"]
    .fillna("")
    .str.contains(r"[^\x00-\x7F]", regex=True)
]

print("S1 rows with non-ASCII business names:", len(non_ascii_s1))

if len(non_ascii_s1) > 0:
    display(
        non_ascii_s1[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country",
            ]
        ].sample(
            n=min(20, len(non_ascii_s1)),
            random_state=42,
        )
    )
else:
    print("No non-ASCII business names found in S1.")

S1 rows with non-ASCII business names: 0
No non-ASCII business names found in S1.


In [9]:
display(
    s1[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
        ]
    ].sample(20, random_state=42)
)

,entity_id,business_name,business_address,country
2195840,S1-53356671,Pediatric Medicine PLLC,"4850 20, Otisco, NY",US
1487051,S1-320151505,Fetech National Twin,"19034 Woodburn Road, Woodburn, IN",US
1878277,S1-938947364,General Design Innovations LLC,"7241 Osage Avenue, Mesa, AZ",US
993966,S1-195839862,Construction Ideaz Papers Private Limited,"Building No.4/606, Prabhul Cottage Karimbalur,...",India
565428,S1-655046555,Penaloza and Bittle First Inc.,"4255 Charleswood Avenue, Memphis, TN",US
1645753,S1-758070915,National Investments LLC,"1641 Virginia Lane, Hueytown, AL",US
1165724,S1-169338169,Reus Idaho Company,"Sanford, ME, 31 Guillemette Street",US
363842,S1-796421498,Creative Projects Limited,"12Thfloor, C, 1204, Roya Oasis, Jankalyan Naga...",India
962445,S1-802535179,Goar Stone LLC,"2105 Wood Ridge Cove, Cedar Park, TX",US
964820,S1-97176033,Lakshmi Consultants Private Limited,"A-301, New Sai Dham Chsl, Ramdev Park Road, Th...",India


In [10]:
import re
import unicodedata


def normalize_text(value):
    if pd.isna(value):
        return ""

    text = str(value)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Case-insensitive normalization
    text = text.casefold()

    # Replace non-alphanumeric characters with spaces.
    # Unicode letters/digits are preserved.
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [11]:
sample_names = s1[
    ["business_name", "business_address"]
].sample(20, random_state=42).copy()

sample_names["normalized_name"] = sample_names["business_name"].apply(
    normalize_text
)

sample_names["normalized_address"] = sample_names["business_address"].apply(
    normalize_text
)

display(sample_names)

,business_name,business_address,normalized_name,normalized_address
2195840,Pediatric Medicine PLLC,"4850 20, Otisco, NY",pediatric medicine pllc,4850 20 otisco ny
1487051,Fetech National Twin,"19034 Woodburn Road, Woodburn, IN",fetech national twin,19034 woodburn road woodburn in
1878277,General Design Innovations LLC,"7241 Osage Avenue, Mesa, AZ",general design innovations llc,7241 osage avenue mesa az
993966,Construction Ideaz Papers Private Limited,"Building No.4/606, Prabhul Cottage Karimbalur,...",construction ideaz papers private limited,building no 4 606 prabhul cottage karimbalur e...
565428,Penaloza and Bittle First Inc.,"4255 Charleswood Avenue, Memphis, TN",penaloza and bittle first inc,4255 charleswood avenue memphis tn
1645753,National Investments LLC,"1641 Virginia Lane, Hueytown, AL",national investments llc,1641 virginia lane hueytown al
1165724,Reus Idaho Company,"Sanford, ME, 31 Guillemette Street",reus idaho company,sanford me 31 guillemette street
363842,Creative Projects Limited,"12Thfloor, C, 1204, Roya Oasis, Jankalyan Naga...",creative projects limited,12thfloor c 1204 roya oasis jankalyan nagarmal...
962445,Goar Stone LLC,"2105 Wood Ridge Cove, Cedar Park, TX",goar stone llc,2105 wood ridge cove cedar park tx
964820,Lakshmi Consultants Private Limited,"A-301, New Sai Dham Chsl, Ramdev Park Road, Th...",lakshmi consultants private limited,a 301 new sai dham chsl ramdev park road thane...


In [12]:
blocking_sample = blocking_sample.copy()

blocking_sample = blocking_sample.merge(
    s1[
        [
            "entity_id",
            "business_name",
            "business_address",
            "country",
        ]
    ],
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left",
)

blocking_sample = blocking_sample.drop(
    columns=["entity_id"]
)

blocking_sample["normalized_name"] = blocking_sample[
    "business_name"
].apply(normalize_text)

print("Blocking sample rows:", len(blocking_sample))
print(
    "Unique normalized names:",
    blocking_sample["normalized_name"].nunique()
)

display(
    blocking_sample[
        [
            "source1_entity_id",
            "business_name",
            "normalized_name",
        ]
    ].head(20)
)

Blocking sample rows: 10000
Unique normalized names: 9824


,source1_entity_id,business_name,normalized_name
0,S1-861482128,Frontier Smart Infinite PLLC,frontier smart infinite pllc
1,S1-411471033,Coastal Fund Ltd,coastal fund ltd
2,S1-694593410,Balaji Agro Private Limited,balaji agro private limited
3,S1-672408263,Floating,floating
4,S1-961865753,Hare Tours (India) Private Limited,hare tours india private limited
5,S1-198211335,Empire Project Center,empire project center
6,S1-594484457,Gypsy Drew Quality Quantumsphere,gypsy drew quality quantumsphere
7,S1-273441004,7 29 Quartzsea-Silver Spring,7 29 quartzsea silver spring
8,S1-320531711,Apex American Ltd,apex american ltd
9,S1-800660172,EB Ventures Private Limited,eb ventures private limited


In [13]:
import time

start = time.time()

s2["normalized_name"] = s2["business_name"].apply(normalize_text)

print(
    f"S2 normalization complete: {time.time() - start:.1f} seconds"
)

print(
    "S2 unique normalized names:",
    s2["normalized_name"].nunique()
)

start = time.time()

s3["normalized_name"] = s3["business_name"].apply(normalize_text)

print(
    f"S3 normalization complete: {time.time() - start:.1f} seconds"
)

print(
    "S3 unique normalized names:",
    s3["normalized_name"].nunique()
)

S2 normalization complete: 16.2 seconds
S2 unique normalized names: 4025057
S3 normalization complete: 16.1 seconds
S3 unique normalized names: 4280400


In [14]:
from collections import defaultdict

s2_name_index = (
    s2.groupby("normalized_name")["entity_id"]
    .agg(list)
    .to_dict()
)

s3_name_index = (
    s3.groupby("normalized_name")["entity_id"]
    .agg(list)
    .to_dict()
)

print("S2 name-index keys:", len(s2_name_index))
print("S3 name-index keys:", len(s3_name_index))

S2 name-index keys: 4025057
S3 name-index keys: 4280400


In [15]:
def get_name_candidates(normalized_name, name_index):
    if not normalized_name:
        return []

    return name_index.get(normalized_name, [])


s1_name_candidates_s2 = {}
s1_name_candidates_s3 = {}

for row in blocking_sample[
    ["source1_entity_id", "normalized_name"]
].itertuples(index=False):

    s1_id = row.source1_entity_id
    normalized_name = row.normalized_name

    s1_name_candidates_s2[s1_id] = get_name_candidates(
        normalized_name,
        s2_name_index,
    )

    s1_name_candidates_s3[s1_id] = get_name_candidates(
        normalized_name,
        s3_name_index,
    )

print("S1 entities:", len(blocking_sample))

print(
    "S1 entities with S2 name candidates:",
    sum(bool(x) for x in s1_name_candidates_s2.values())
)

print(
    "S1 entities with S3 name candidates:",
    sum(bool(x) for x in s1_name_candidates_s3.values())
)

S1 entities: 10000
S1 entities with S2 name candidates: 5336
S1 entities with S3 name candidates: 5530


In [16]:
s2_candidate_counts = np.array(
    [len(x) for x in s1_name_candidates_s2.values()]
)

s3_candidate_counts = np.array(
    [len(x) for x in s1_name_candidates_s3.values()]
)

print("S2 candidate statistics:")
print(
    pd.Series(s2_candidate_counts).describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

print("\nS3 candidate statistics:")
print(
    pd.Series(s3_candidate_counts).describe(
        percentiles=[0.50, 0.90, 0.95, 0.99]
    )
)

S2 candidate statistics:
count    10000.000000
mean         4.626000
std         15.305808
min          0.000000
50%          1.000000
90%          6.000000
95%         29.000000
99%         88.020000
max        224.000000
dtype: float64

S3 candidate statistics:
count    10000.000000
mean         5.087200
std         15.634706
min          0.000000
50%          1.000000
90%         10.000000
95%         30.000000
99%         88.000000
max        235.000000
dtype: float64


In [17]:
def parse_gt_ids(value):
    if pd.isna(value) or value == "":
        return []

    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip()
    ]


name_block_results = []

for row in blocking_sample[
    [
        "source1_entity_id",
        "matched_entity_ids",
    ]
].itertuples(index=False):

    s1_id = row.source1_entity_id

    gt_ids = set(parse_gt_ids(row.matched_entity_ids))

    s2_candidates = set(
        s1_name_candidates_s2.get(s1_id, [])
    )

    s3_candidates = set(
        s1_name_candidates_s3.get(s1_id, [])
    )

    candidates = s2_candidates | s3_candidates

    matched_gt = gt_ids & candidates

    name_block_results.append(
        {
            "source1_entity_id": s1_id,
            "gt_count": len(gt_ids),
            "candidate_count": len(candidates),
            "gt_recovered": len(matched_gt),
            "all_gt_recovered": (
                len(gt_ids) == len(matched_gt)
            ),
        }
    )

name_block_results = pd.DataFrame(
    name_block_results
)

print(
    "Total ground-truth matches:",
    name_block_results["gt_count"].sum()
)

print(
    "Ground-truth matches recovered:",
    name_block_results["gt_recovered"].sum()
)

print(
    "Pair-level blocking recall:",
    (
        name_block_results["gt_recovered"].sum()
        / name_block_results["gt_count"].sum()
    )
)

print(
    "S1 entities with ALL true matches recovered:",
    name_block_results["all_gt_recovered"].sum()
)

print(
    "S1-level complete recall:",
    name_block_results["all_gt_recovered"].mean()
)

Total ground-truth matches: 34609
Ground-truth matches recovered: 7531
Pair-level blocking recall: 0.21760235776821057
S1 entities with ALL true matches recovered: 833
S1-level complete recall: 0.0833


In [18]:
import time

start = time.time()

s2["normalized_address"] = s2[
    "business_address"
].apply(normalize_text)

print(
    f"S2 address normalization complete: "
    f"{time.time() - start:.1f} seconds"
)

print(
    "S2 unique normalized addresses:",
    s2["normalized_address"].nunique()
)

start = time.time()

s3["normalized_address"] = s3[
    "business_address"
].apply(normalize_text)

print(
    f"S3 address normalization complete: "
    f"{time.time() - start:.1f} seconds"
)

print(
    "S3 unique normalized addresses:",
    s3["normalized_address"].nunique()
)

S2 address normalization complete: 23.6 seconds
S2 unique normalized addresses: 4286078
S3 address normalization complete: 25.5 seconds
S3 unique normalized addresses: 4616045


In [19]:
s2_address_index = (
    s2.groupby("normalized_address")["entity_id"]
    .agg(list)
    .to_dict()
)

s3_address_index = (
    s3.groupby("normalized_address")["entity_id"]
    .agg(list)
    .to_dict()
)

print(
    "S2 address-index keys:",
    len(s2_address_index)
)

print(
    "S3 address-index keys:",
    len(s3_address_index)
)

S2 address-index keys: 4286078
S3 address-index keys: 4616045


In [22]:
print(blocking_sample.columns.tolist())

['source1_entity_id', 'matched_entity_ids', 'match_count', 'business_name', 'business_address', 'country', 'normalized_name']


In [23]:
blocking_sample["normalized_name_v2"] = (
    blocking_sample["business_name"].apply(normalize_name_v2)
)

blocking_sample["normalized_address_v2"] = (
    blocking_sample["business_address"].apply(normalize_address_v2)
)

print(
    blocking_sample[
        [
            "source1_entity_id",
            "normalized_name_v2",
            "normalized_address_v2"
        ]
    ].head()
)

NameError: name 'normalize_name_v2' is not defined

In [21]:
s1_address_candidates_s2 = {}
s1_address_candidates_s3 = {}

for row in blocking_sample[
    ["source1_entity_id", "normalized_address_v2"]
].itertuples(index=False):

    s1_id = row.source1_entity_id
    normalized_address = row.normalized_address_v2

    if normalized_address:
        s1_address_candidates_s2[s1_id] = (
            s2_address_index.get(normalized_address, [])
        )

        s1_address_candidates_s3[s1_id] = (
            s3_address_index.get(normalized_address, [])
        )
    else:
        s1_address_candidates_s2[s1_id] = []
        s1_address_candidates_s3[s1_id] = []

print("S1 entities:", len(blocking_sample))

print(
    "S1 entities with S2 address candidates:",
    sum(bool(x) for x in s1_address_candidates_s2.values())
)

print(
    "S1 entities with S3 address candidates:",
    sum(bool(x) for x in s1_address_candidates_s3.values())
)

KeyError: "['normalized_address_v2'] not in index"

In [ ]:
blocking_sample["normalized_address"] = blocking_sample[
    "business_address"
].apply(normalize_text)

print(
    "S1 normalized addresses:",
    blocking_sample["normalized_address"].notna().sum()
)

print(
    "Unique normalized S1 addresses:",
    blocking_sample["normalized_address"].nunique()
)

S1 normalized addresses: 10000
Unique normalized S1 addresses: 9997


In [ ]:
s1_address_candidates_s2 = {}
s1_address_candidates_s3 = {}

for row in blocking_sample[
    ["source1_entity_id", "normalized_address"]
].itertuples(index=False):

    s1_id = row.source1_entity_id
    normalized_address = row.normalized_address

    if normalized_address:
        s1_address_candidates_s2[s1_id] = (
            s2_address_index.get(normalized_address, [])
        )

        s1_address_candidates_s3[s1_id] = (
            s3_address_index.get(normalized_address, [])
        )
    else:
        s1_address_candidates_s2[s1_id] = []
        s1_address_candidates_s3[s1_id] = []

print("S1 entities:", len(blocking_sample))

print(
    "S1 entities with S2 address candidates:",
    sum(bool(x) for x in s1_address_candidates_s2.values())
)

print(
    "S1 entities with S3 address candidates:",
    sum(bool(x) for x in s1_address_candidates_s3.values())
)

S1 entities: 10000
S1 entities with S2 address candidates: 1827
S1 entities with S3 address candidates: 885


In [ ]:
address_block_results = []

for row in blocking_sample[
    [
        "source1_entity_id",
        "matched_entity_ids",
    ]
].itertuples(index=False):

    s1_id = row.source1_entity_id

    gt_ids = set(
        parse_gt_ids(row.matched_entity_ids)
    )

    s2_candidates = set(
        s1_address_candidates_s2.get(s1_id, [])
    )

    s3_candidates = set(
        s1_address_candidates_s3.get(s1_id, [])
    )

    candidates = s2_candidates | s3_candidates

    matched_gt = gt_ids & candidates

    address_block_results.append(
        {
            "source1_entity_id": s1_id,
            "gt_count": len(gt_ids),
            "candidate_count": len(candidates),
            "gt_recovered": len(matched_gt),
            "all_gt_recovered": (
                len(gt_ids) == len(matched_gt)
            ),
        }
    )

address_block_results = pd.DataFrame(
    address_block_results
)

total_gt = address_block_results["gt_count"].sum()
recovered_gt = address_block_results["gt_recovered"].sum()

print(
    "Total ground-truth matches:",
    total_gt
)

print(
    "Ground-truth matches recovered:",
    recovered_gt
)

print(
    "Pair-level address blocking recall:",
    recovered_gt / total_gt
)

print(
    "S1 entities with ALL true matches recovered:",
    address_block_results["all_gt_recovered"].sum()
)

print(
    "S1-level complete recall:",
    address_block_results["all_gt_recovered"].mean()
)

Total ground-truth matches: 34609
Ground-truth matches recovered: 2955
Pair-level address blocking recall: 0.08538241497876276
S1 entities with ALL true matches recovered: 619
S1-level complete recall: 0.0619


In [ ]:
combined_block_results = []

for row in blocking_sample[
    [
        "source1_entity_id",
        "matched_entity_ids",
    ]
].itertuples(index=False):

    s1_id = row.source1_entity_id

    gt_ids = set(
        parse_gt_ids(row.matched_entity_ids)
    )

    # Name candidates
    name_candidates = (
        set(s1_name_candidates_s2.get(s1_id, []))
        | set(s1_name_candidates_s3.get(s1_id, []))
    )

    # Address candidates
    address_candidates = (
        set(s1_address_candidates_s2.get(s1_id, []))
        | set(s1_address_candidates_s3.get(s1_id, []))
    )

    # Union of both blocks
    combined_candidates = (
        name_candidates | address_candidates
    )

    # True matches recovered
    matched_gt = gt_ids & combined_candidates

    combined_block_results.append(
        {
            "source1_entity_id": s1_id,
            "gt_count": len(gt_ids),
            "name_candidate_count": len(name_candidates),
            "address_candidate_count": len(address_candidates),
            "combined_candidate_count": len(combined_candidates),
            "gt_recovered": len(matched_gt),
            "all_gt_recovered": (
                len(gt_ids) == len(matched_gt)
            ),
            "new_gt_from_address": len(
                gt_ids
                & address_candidates
                - name_candidates
            ),
        }
    )

combined_block_results = pd.DataFrame(
    combined_block_results
)

total_gt = combined_block_results["gt_count"].sum()
recovered_gt = combined_block_results["gt_recovered"].sum()

print("Total ground-truth matches:", total_gt)
print("Ground-truth matches recovered:", recovered_gt)

print(
    "Combined pair-level blocking recall:",
    recovered_gt / total_gt
)

print(
    "S1 entities with ALL true matches recovered:",
    combined_block_results["all_gt_recovered"].sum()
)

print(
    "Combined S1-level complete recall:",
    combined_block_results["all_gt_recovered"].mean()
)

print(
    "Additional true pairs recovered by address:",
    combined_block_results["new_gt_from_address"].sum()
)

print(
    "Additional recall contributed by address:",
    combined_block_results["new_gt_from_address"].sum()
    / total_gt
)

Total ground-truth matches: 34609
Ground-truth matches recovered: 9968
Combined pair-level blocking recall: 0.28801756768470627
S1 entities with ALL true matches recovered: 989
Combined S1-level complete recall: 0.0989
Additional true pairs recovered by address: 2437
Additional recall contributed by address: 0.0704152099164957


In [ ]:
from collections import Counter

s2_name_token_counts = Counter()

for name in s2["normalized_name"]:
    if not name:
        continue

    tokens = set(name.split())

    s2_name_token_counts.update(tokens)

print(
    "Unique S2 name tokens:",
    len(s2_name_token_counts)
)

print("\nMost common tokens:")

for token, count in s2_name_token_counts.most_common(30):
    print(f"{token:<25} {count:>10,}")

Unique S2 name tokens: 810514

Most common tokens:
limited                      528,795
llc                          528,303
private                      528,131
inc                          400,229
ltd                          398,464
ल                            237,102
ट                            232,862
र                            226,825
ड                            208,515
प                            206,868
म                            206,488
com                          201,464
center                       193,696
pvt                          189,270
स                            160,325
इव                           157,955
partners                     152,694
services                     145,239
corp                         141,773
co                           139,561
group                        139,390
s                            138,303
and                          118,201
c                            113,281
india                        110,684
क                       

In [ ]:
token_frequency_series = pd.Series(
    list(s2_name_token_counts.values())
)

print("Total unique tokens:", len(token_frequency_series))

print("\nFrequency distribution:")
print(
    token_frequency_series.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999,
        ]
    )
)

print("\nTokens with frequency <= threshold:")

for threshold in [
    1,
    2,
    5,
    10,
    25,
    50,
    100,
    500,
    1000,
    5000,
    10000,
    50000,
]:
    print(
        f"{threshold:>6,}: "
        f"{(token_frequency_series <= threshold).sum():>8,} tokens"
    )

Total unique tokens: 810514

Frequency distribution:
count    810514.000000
mean         25.837501
std        1570.054957
min           1.000000
50%           1.000000
75%           1.000000
90%           6.000000
95%          18.000000
99%         114.000000
99.5%       275.000000
99.9%      3034.948000
max      528795.000000
dtype: float64

Tokens with frequency <= threshold:
     1:  640,329 tokens
     2:  690,116 tokens
     5:  728,996 tokens
    10:  753,297 tokens
    25:  779,177 tokens
    50:  791,406 tokens
   100:  801,413 tokens
   500:  807,680 tokens
 1,000:  808,972 tokens
 5,000:  809,965 tokens
10,000:  810,176 tokens
50,000:  810,474 tokens


In [ ]:
MAX_TOKEN_FREQUENCY = 5000

s2_informative_tokens = {
    token
    for token, count in s2_name_token_counts.items()
    if count <= MAX_TOKEN_FREQUENCY
}

print(
    "Informative S2 tokens:",
    len(s2_informative_tokens)
)

print(
    "Excluded S2 tokens:",
    len(s2_name_token_counts) - len(s2_informative_tokens)
)

Informative S2 tokens: 809965
Excluded S2 tokens: 549


In [ ]:
s2_token_index = defaultdict(list)

for entity_id, normalized_name in zip(
    s2["entity_id"],
    s2["normalized_name"],
):

    if not normalized_name:
        continue

    tokens = set(normalized_name.split())

    for token in tokens:
        if token in s2_informative_tokens:
            s2_token_index[token].append(entity_id)

print(
    "S2 informative-token index keys:",
    len(s2_token_index)
)

S2 informative-token index keys: 809965


In [ ]:
TOKEN_FREQUENCY_THRESHOLDS = [
    50,
    100,
    500,
    1000,
    5000,
]

# Keep only tokens whose target-side frequency
# is <= the current threshold.
s2_token_frequency = s2_name_token_counts

token_block_summary = []

for threshold in TOKEN_FREQUENCY_THRESHOLDS:

    candidate_counts = []
    recovered_pairs = 0
    total_pairs = 0

    for row in blocking_sample[
        ["source1_entity_id", "normalized_name", "matched_entity_ids"]
    ].itertuples(index=False):

        gt_ids = set(
            parse_gt_ids(row.matched_entity_ids)
        )

        total_pairs += len(gt_ids)

        if not row.normalized_name:
            candidate_ids = set()
        else:
            tokens = set(row.normalized_name.split())

            candidate_ids = set()

            for token in tokens:

                if s2_token_frequency.get(token, 0) <= threshold:

                    candidate_ids.update(
                        s2_token_index.get(token, [])
                    )

        candidate_counts.append(len(candidate_ids))

        recovered_pairs += len(
            gt_ids & candidate_ids
        )

    token_block_summary.append(
        {
            "threshold": threshold,
            "pair_recall": (
                recovered_pairs / total_pairs
            ),
            "mean_candidates": np.mean(candidate_counts),
            "median_candidates": np.median(candidate_counts),
            "p90_candidates": np.percentile(
                candidate_counts, 90
            ),
            "p95_candidates": np.percentile(
                candidate_counts, 95
            ),
            "p99_candidates": np.percentile(
                candidate_counts, 99
            ),
            "max_candidates": np.max(candidate_counts),
        }
    )

token_block_summary = pd.DataFrame(
    token_block_summary
)

display(token_block_summary)

,threshold,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates
0,50,0.082493,4.4768,0.0,20.0,35.0,49.00,94
1,100,0.137016,15.5623,0.0,67.0,85.0,107.00,198
2,500,0.203849,65.8538,0.0,229.0,339.0,506.01,1008
3,1000,0.243029,246.8581,27.0,895.0,997.0,1472.01,2419
4,5000,0.319223,1550.9120,866.0,4190.0,4881.0,7712.08,12895


In [ ]:
combined_token_summary = []

for threshold in TOKEN_FREQUENCY_THRESHOLDS:

    total_pairs = 0
    recovered_pairs = 0

    candidate_counts = []
    complete_recall_count = 0

    for row in blocking_sample[
        [
            "source1_entity_id",
            "normalized_name",
            "matched_entity_ids",
        ]
    ].itertuples(index=False):

        s1_id = row.source1_entity_id

        gt_ids = set(
            parse_gt_ids(row.matched_entity_ids)
        )

        total_pairs += len(gt_ids)

        # -------------------------------------------------
        # Block A: exact normalized name
        # -------------------------------------------------
        name_candidates = (
            set(
                s1_name_candidates_s2.get(
                    s1_id, []
                )
            )
            |
            set(
                s1_name_candidates_s3.get(
                    s1_id, []
                )
            )
        )

        # -------------------------------------------------
        # Block B: exact normalized address
        # -------------------------------------------------
        address_candidates = (
            set(
                s1_address_candidates_s2.get(
                    s1_id, []
                )
            )
            |
            set(
                s1_address_candidates_s3.get(
                    s1_id, []
                )
            )
        )

        # -------------------------------------------------
        # Block C: informative name tokens
        # -------------------------------------------------
        token_candidates = set()

        if row.normalized_name:

            tokens = set(
                row.normalized_name.split()
            )

            for token in tokens:

                if (
                    s2_token_frequency.get(
                        token, 0
                    ) <= threshold
                ):
                    token_candidates.update(
                        s2_token_index.get(
                            token, []
                        )
                    )

        # -------------------------------------------------
        # Union all blocks
        # -------------------------------------------------
        combined_candidates = (
            name_candidates
            |
            address_candidates
            |
            token_candidates
        )

        recovered = (
            gt_ids & combined_candidates
        )

        recovered_pairs += len(recovered)

        candidate_counts.append(
            len(combined_candidates)
        )

        if len(gt_ids) == len(recovered):
            complete_recall_count += 1

    combined_token_summary.append(
        {
            "token_threshold": threshold,
            "pair_recall": (
                recovered_pairs / total_pairs
            ),
            "mean_candidates": np.mean(
                candidate_counts
            ),
            "median_candidates": np.median(
                candidate_counts
            ),
            "p90_candidates": np.percentile(
                candidate_counts, 90
            ),
            "p95_candidates": np.percentile(
                candidate_counts, 95
            ),
            "p99_candidates": np.percentile(
                candidate_counts, 99
            ),
            "max_candidates": np.max(
                candidate_counts
            ),
            "complete_s1_recall": (
                complete_recall_count
                / len(blocking_sample)
            ),
        }
    )

combined_token_summary = pd.DataFrame(
    combined_token_summary
)

display(combined_token_summary)

,token_threshold,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,50,0.341472,14.3725,3.0,41.0,61.00,176.01,460,0.1181
1,100,0.376694,25.3413,6.0,79.0,95.00,179.00,460,0.1313
2,500,0.419428,75.4944,17.0,237.0,341.00,507.02,1009,0.1487
3,1000,0.442515,256.0740,56.0,897.0,1006.00,1472.02,2419,0.1569
4,5000,0.491086,1558.6404,866.5,4196.1,4886.05,7720.01,12897,0.1716


In [ ]:
s3_name_token_counts = Counter()

for name in s3["normalized_name"]:
    if not name:
        continue

    tokens = set(name.split())

    s3_name_token_counts.update(tokens)

print(
    "Unique S3 name tokens:",
    len(s3_name_token_counts)
)

print("\nMost common S3 tokens:")

for token, count in s3_name_token_counts.most_common(30):
    print(f"{token:<25} {count:>10,}")

Unique S3 name tokens: 863014

Most common S3 tokens:
limited                      673,480
private                      639,861
llc                          571,362
ltd                          427,781
inc                          419,917
center                       220,809
pvt                          220,353
com                          211,116
services                     166,621
partners                     161,955
s                            147,647
group                        144,614
co                           143,838
corp                         140,236
ल                            134,117
ट                            132,429
र                            128,844
and                          122,220
ड                            117,653
प                            116,694
म                            116,370
c                            113,132
india                        112,959
holdings                      99,024
care                          95,821
of                   

In [ ]:
s3_token_frequency_series = pd.Series(
    list(s3_name_token_counts.values())
)

print(
    "Total unique S3 tokens:",
    len(s3_token_frequency_series)
)

print("\nFrequency distribution:")

print(
    s3_token_frequency_series.describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995,
            0.999,
        ]
    )
)

print("\nTokens with frequency <= threshold:")

for threshold in [
    1,
    2,
    5,
    10,
    25,
    50,
    100,
    500,
    1000,
    5000,
    10000,
    50000,
]:
    print(
        f"{threshold:>6,}: "
        f"{(s3_token_frequency_series <= threshold).sum():>8,} tokens"
    )

Total unique S3 tokens: 863014

Frequency distribution:
count    863014.000000
mean         24.081057
std        1593.252996
min           1.000000
50%           1.000000
75%           1.000000
90%           5.000000
95%          17.000000
99%         109.000000
99.5%       254.000000
99.9%      2394.844000
max      673480.000000
dtype: float64

Tokens with frequency <= threshold:
     1:  673,612 tokens
     2:  734,658 tokens
     5:  779,246 tokens
    10:  804,153 tokens
    25:  829,036 tokens
    50:  841,363 tokens
   100:  853,517 tokens
   500:  860,216 tokens
 1,000:  861,464 tokens
 5,000:  862,459 tokens
10,000:  862,668 tokens
50,000:  862,972 tokens


In [ ]:
MAX_TOKEN_FREQUENCY = 5000

s3_informative_tokens = {
    token
    for token, count in s3_name_token_counts.items()
    if count <= MAX_TOKEN_FREQUENCY
}

print(
    "Informative S3 tokens:",
    len(s3_informative_tokens)
)

print(
    "Excluded S3 tokens:",
    len(s3_name_token_counts)
    - len(s3_informative_tokens)
)

Informative S3 tokens: 862459
Excluded S3 tokens: 555


In [ ]:
s3_token_index = defaultdict(list)

for entity_id, normalized_name in zip(
    s3["entity_id"],
    s3["normalized_name"],
):

    if not normalized_name:
        continue

    tokens = set(normalized_name.split())

    for token in tokens:
        if token in s3_informative_tokens:
            s3_token_index[token].append(entity_id)

print(
    "S3 informative-token index keys:",
    len(s3_token_index)
)

S3 informative-token index keys: 862459


In [ ]:
TOKEN_FREQUENCY_THRESHOLDS = [
    50,
    100,
    500,
    1000,
    5000,
]

token_block_both_summary = []

for threshold in TOKEN_FREQUENCY_THRESHOLDS:

    total_pairs = 0
    recovered_pairs = 0
    candidate_counts = []
    complete_recall_count = 0

    for row in blocking_sample[
        [
            "source1_entity_id",
            "normalized_name",
            "matched_entity_ids",
        ]
    ].itertuples(index=False):

        s1_id = row.source1_entity_id

        gt_ids = set(
            parse_gt_ids(row.matched_entity_ids)
        )

        total_pairs += len(gt_ids)

        token_candidates = set()

        if row.normalized_name:

            tokens = set(
                row.normalized_name.split()
            )

            for token in tokens:

                # S2
                if (
                    s2_token_frequency.get(
                        token, 0
                    ) <= threshold
                ):
                    token_candidates.update(
                        s2_token_index.get(
                            token, []
                        )
                    )

                # S3
                if (
                    s3_name_token_counts.get(
                        token, 0
                    ) <= threshold
                ):
                    token_candidates.update(
                        s3_token_index.get(
                            token, []
                        )
                    )

        recovered = gt_ids & token_candidates

        recovered_pairs += len(recovered)
        candidate_counts.append(
            len(token_candidates)
        )

        if len(gt_ids) == len(recovered):
            complete_recall_count += 1

    token_block_both_summary.append(
        {
            "token_threshold": threshold,
            "pair_recall": (
                recovered_pairs / total_pairs
            ),
            "mean_candidates": np.mean(
                candidate_counts
            ),
            "median_candidates": np.median(
                candidate_counts
            ),
            "p90_candidates": np.percentile(
                candidate_counts,
                90
            ),
            "p95_candidates": np.percentile(
                candidate_counts,
                95
            ),
            "p99_candidates": np.percentile(
                candidate_counts,
                99
            ),
            "max_candidates": np.max(
                candidate_counts
            ),
            "complete_s1_recall": (
                complete_recall_count
                / len(blocking_sample)
            ),
        }
    )

token_block_both_summary = pd.DataFrame(
    token_block_both_summary
)

display(token_block_both_summary)

,token_threshold,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,50,0.168511,8.8018,0.0,42.0,61.00,91.00,156,0.1666
1,100,0.281516,30.9880,0.0,129.0,164.00,207.00,407,0.2473
2,500,0.419775,132.6122,0.0,466.0,675.05,999.04,2067,0.3558
3,1000,0.496518,483.3786,54.0,1761.0,1961.00,2819.04,4328,0.3998
4,5000,0.641481,2732.1829,1772.5,7583.2,9307.10,14152.35,26152,0.4922


In [ ]:
all_blocks_summary = []

for threshold in TOKEN_FREQUENCY_THRESHOLDS:

    total_pairs = 0
    recovered_pairs = 0

    candidate_counts = []
    complete_recall_count = 0

    for row in blocking_sample[
        [
            "source1_entity_id",
            "normalized_name",
            "matched_entity_ids",
        ]
    ].itertuples(index=False):

        s1_id = row.source1_entity_id

        gt_ids = set(
            parse_gt_ids(row.matched_entity_ids)
        )

        total_pairs += len(gt_ids)

        # ---------------------------------------------
        # Block 1: exact normalized name
        # ---------------------------------------------
        name_candidates = (
            set(
                s1_name_candidates_s2.get(
                    s1_id, []
                )
            )
            |
            set(
                s1_name_candidates_s3.get(
                    s1_id, []
                )
            )
        )

        # ---------------------------------------------
        # Block 2: exact normalized address
        # ---------------------------------------------
        address_candidates = (
            set(
                s1_address_candidates_s2.get(
                    s1_id, []
                )
            )
            |
            set(
                s1_address_candidates_s3.get(
                    s1_id, []
                )
            )
        )

        # ---------------------------------------------
        # Block 3 + 4: informative name tokens
        # ---------------------------------------------
        s2_token_candidates = set()
        s3_token_candidates = set()

        if row.normalized_name:

            tokens = set(
                row.normalized_name.split()
            )

            for token in tokens:

                # S2
                if (
                    s2_token_frequency.get(
                        token, 0
                    ) <= threshold
                ):
                    s2_token_candidates.update(
                        s2_token_index.get(
                            token, []
                        )
                    )

                # S3
                if (
                    s3_name_token_counts.get(
                        token, 0
                    ) <= threshold
                ):
                    s3_token_candidates.update(
                        s3_token_index.get(
                            token, []
                        )
                    )

        # ---------------------------------------------
        # Union all blocks
        # ---------------------------------------------
        combined_candidates = (
            name_candidates
            |
            address_candidates
            |
            s2_token_candidates
            |
            s3_token_candidates
        )

        recovered = (
            gt_ids & combined_candidates
        )

        recovered_pairs += len(recovered)

        candidate_counts.append(
            len(combined_candidates)
        )

        if len(gt_ids) == len(recovered):
            complete_recall_count += 1

    all_blocks_summary.append(
        {
            "token_threshold": threshold,
            "pair_recall": (
                recovered_pairs / total_pairs
            ),
            "mean_candidates": np.mean(
                candidate_counts
            ),
            "median_candidates": np.median(
                candidate_counts
            ),
            "p90_candidates": np.percentile(
                candidate_counts,
                90
            ),
            "p95_candidates": np.percentile(
                candidate_counts,
                95
            ),
            "p99_candidates": np.percentile(
                candidate_counts,
                99
            ),
            "max_candidates": np.max(
                candidate_counts
            ),
            "complete_s1_recall": (
                complete_recall_count
                / len(blocking_sample)
            ),
        }
    )

all_blocks_summary = pd.DataFrame(
    all_blocks_summary
)

display(all_blocks_summary)


,token_threshold,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,50,0.402034,18.5882,3.0,61.0,84.00,176.01,460,0.2077
1,100,0.478806,40.5521,8.0,139.0,172.00,236.01,460,0.2848
2,500,0.570863,141.9009,23.0,467.1,675.05,999.04,2067,0.3878
3,1000,0.617065,491.9874,87.0,1761.1,1961.00,2819.04,4332,0.4291
4,5000,0.711925,2737.8840,1772.5,7583.2,9307.10,14152.35,26152,0.5174


In [ ]:
s2_address_token_counts = Counter()

for address in s2["normalized_address"]:
    if not address:
        continue

    tokens = set(address.split())

    s2_address_token_counts.update(tokens)

print(
    "Unique S2 address tokens:",
    len(s2_address_token_counts)
)

print("\nMost common S2 address tokens:")

for token, count in s2_address_token_counts.most_common(30):
    print(f"{token:<25} {count:>10,}")

Unique S2 address tokens: 607943

Most common S2 address tokens:
no                           916,456
road                         669,292
street                       337,310
st                           333,485
rd                           330,958
floor                        319,586
maharashtra                  319,026
city                         298,415
tx                           292,289
nagar                        286,965
dr                           283,537
1                            281,615
a                            265,465
delhi                        262,478
c                            254,392
2                            228,483
ny                           225,422
ave                          224,070
drive                        215,965
pradesh                      214,708
new                          210,472
nc                           207,687
west                         200,360
avenue                       193,632
oh                           187,361
h         

In [ ]:
address_freq_series = pd.Series(
    s2_address_token_counts,
    dtype="int64"
)

print(address_freq_series.describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]
))

print("\nTokens at frequency thresholds:")

for threshold in [1, 2, 5, 10, 25, 50, 100, 500, 1000, 5000, 10000, 50000]:
    count = (address_freq_series <= threshold).sum()
    print(
        f"<= {threshold:>5}: "
        f"{count:>8,} / {len(address_freq_series):,}"
    )

count    607943.000000
mean         64.634711
std        2546.423968
min           1.000000
50%           2.000000
75%           6.000000
90%          22.000000
95%          61.000000
99%         502.000000
99.5%      1094.000000
99.9%      6978.348000
max      916456.000000
dtype: float64

Tokens at frequency thresholds:
<=     1:  197,587 / 607,943
<=     2:  309,011 / 607,943
<=     5:  450,743 / 607,943
<=    10:  508,067 / 607,943
<=    25:  553,212 / 607,943
<=    50:  573,244 / 607,943
<=   100:  586,632 / 607,943
<=   500:  601,853 / 607,943
<=  1000:  604,651 / 607,943
<=  5000:  607,152 / 607,943
<= 10000:  607,503 / 607,943
<= 50000:  607,825 / 607,943


In [ ]:
from collections import defaultdict

ADDRESS_TOKEN_THRESHOLDS = [10, 25, 50, 100, 500, 1000, 5000]

# Build S2 address-token index.
# Only keep tokens whose corpus frequency is <= 5000.
s2_address_token_index = defaultdict(list)

for entity_id, address in zip(
    s2["entity_id"],
    s2["normalized_address"]
):
    if not address:
        continue

    tokens = set(address.split())

    for token in tokens:
        if s2_address_token_counts.get(token, 0) <= 5000:
            s2_address_token_index[token].append(entity_id)

print("Address-token index keys:", len(s2_address_token_index))

Address-token index keys: 607152


In [ ]:
address_block_summary_s2 = []

for threshold in ADDRESS_TOKEN_THRESHOLDS:
    total_pairs = 0
    recovered_pairs = 0
    candidate_counts = []
    complete_recall_count = 0

    for row in blocking_sample[
        ["source1_entity_id", "normalized_address", "matched_entity_ids"]
    ].itertuples(index=False):

        s1_id = row.source1_entity_id
        gt_ids = set(parse_gt_ids(row.matched_entity_ids))

        total_pairs += len(gt_ids)

        candidates = set()

        if row.normalized_address:
            tokens = set(row.normalized_address.split())

            for token in tokens:
                if s2_address_token_counts.get(token, 0) <= threshold:
                    candidates.update(
                        s2_address_token_index.get(token, [])
                    )

        recovered = gt_ids & candidates

        recovered_pairs += len(recovered)
        candidate_counts.append(len(candidates))

        if len(gt_ids) == len(recovered):
            complete_recall_count += 1

    address_block_summary_s2.append({
        "address_token_threshold": threshold,
        "pair_recall": recovered_pairs / total_pairs,
        "mean_candidates": np.mean(candidate_counts),
        "median_candidates": np.median(candidate_counts),
        "p90_candidates": np.percentile(candidate_counts, 90),
        "p95_candidates": np.percentile(candidate_counts, 95),
        "p99_candidates": np.percentile(candidate_counts, 99),
        "max_candidates": np.max(candidate_counts),
        "complete_s1_recall": (
            complete_recall_count / len(blocking_sample)
        ),
    })

address_block_summary_s2 = pd.DataFrame(address_block_summary_s2)

display(address_block_summary_s2)


,address_token_threshold,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,10,0.081944,0.9328,0.0,4.0,7.00,10.00,24,0.0672
1,25,0.121645,3.1204,0.0,13.0,19.05,28.00,60,0.0712
2,50,0.154757,7.6569,0.0,31.0,43.00,61.00,121,0.0759
3,100,0.198128,20.9061,0.0,77.0,93.00,137.00,258,0.0818
4,500,0.332255,204.6385,136.0,500.0,635.00,917.08,1922,0.0997
5,1000,0.382386,519.7206,400.0,1178.0,1471.00,2024.04,3383,0.1055
6,5000,0.444075,3433.6842,2918.0,7277.2,8792.40,12368.48,24358,0.1147


In [ ]:
s3_address_token_counts = Counter()

for address in s3["normalized_address"]:
    if not address:
        continue

    tokens = set(address.split())
    s3_address_token_counts.update(tokens)

print(
    "Unique S3 address tokens:",
    len(s3_address_token_counts)
)

print("\nMost common S3 address tokens:")

for token, count in s3_address_token_counts.most_common(30):
    print(f"{token:<25} {count:>10,}")

Unique S3 address tokens: 580065

Most common S3 address tokens:
no                           911,355
road                         619,124
new                          466,739
street                       357,143
city                         352,105
st                           331,436
north                        327,168
rd                           326,238
mh                           318,420
1                            309,522
texas                        293,313
dr                           281,587
floor                        277,745
delhi                        272,877
a                            270,451
nagar                        267,813
c                            259,963
drive                        241,783
2                            238,183
york                         228,655
ave                          224,220
carolina                     215,875
avenue                       212,211
dl                           201,529
mumbai                       197,983
unit      

In [ ]:
s3_address_token_index = defaultdict(list)

for entity_id, address in zip(
    s3["entity_id"],
    s3["normalized_address"]
):
    if not address:
        continue

    tokens = set(address.split())

    for token in tokens:
        if s3_address_token_counts.get(token, 0) <= 5000:
            s3_address_token_index[token].append(entity_id)

print(
    "S3 address-token index keys:",
    len(s3_address_token_index)
)

S3 address-token index keys: 579246


In [ ]:
address_block_summary_s3 = []

for threshold in ADDRESS_TOKEN_THRESHOLDS:
    total_pairs = 0
    recovered_pairs = 0
    candidate_counts = []
    complete_recall_count = 0

    for row in blocking_sample[
        ["source1_entity_id", "normalized_address", "matched_entity_ids"]
    ].itertuples(index=False):

        s1_id = row.source1_entity_id
        gt_ids = set(parse_gt_ids(row.matched_entity_ids))

        total_pairs += len(gt_ids)

        candidates = set()

        if row.normalized_address:
            tokens = set(row.normalized_address.split())

            for token in tokens:
                if s3_address_token_counts.get(token, 0) <= threshold:
                    candidates.update(
                        s3_address_token_index.get(token, [])
                    )

        recovered = gt_ids & candidates

        recovered_pairs += len(recovered)
        candidate_counts.append(len(candidates))

        if len(gt_ids) == len(recovered):
            complete_recall_count += 1

    address_block_summary_s3.append({
        "address_token_threshold": threshold,
        "pair_recall": recovered_pairs / total_pairs,
        "mean_candidates": np.mean(candidate_counts),
        "median_candidates": np.median(candidate_counts),
        "p90_candidates": np.percentile(candidate_counts, 90),
        "p95_candidates": np.percentile(candidate_counts, 95),
        "p99_candidates": np.percentile(candidate_counts, 99),
        "max_candidates": np.max(candidate_counts),
        "complete_s1_recall": (
            complete_recall_count / len(blocking_sample)
        ),
    })

address_block_summary_s3 = pd.DataFrame(address_block_summary_s3)

display(address_block_summary_s3)

,address_token_threshold,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,10,0.080933,0.9429,0.0,4.0,7.00,10.00,20,0.0661
1,25,0.118900,3.0488,0.0,13.0,20.00,27.01,51,0.0717
2,50,0.154960,7.8870,0.0,33.0,44.00,64.00,123,0.0780
3,100,0.197434,20.9909,0.0,76.0,93.00,141.00,297,0.0831
4,500,0.333757,204.2388,133.0,503.1,644.05,913.02,1557,0.1008
5,1000,0.388078,513.6154,390.0,1167.1,1455.05,2035.02,3738,0.1080
6,5000,0.461354,4180.8678,3762.5,8547.3,10138.90,13703.12,27093,0.1189


In [ ]:
RAREST_K_VALUES = [1, 2, 3]

rarest_address_block_s2 = []

for k in RAREST_K_VALUES:
    total_pairs = 0
    recovered_pairs = 0
    candidate_counts = []
    complete_recall_count = 0

    for row in blocking_sample[
        ["source1_entity_id", "normalized_address", "matched_entity_ids"]
    ].itertuples(index=False):

        gt_ids = set(parse_gt_ids(row.matched_entity_ids))
        total_pairs += len(gt_ids)

        candidates = set()

        if row.normalized_address:
            tokens = set(row.normalized_address.split())

            # Keep only tokens that exist in the frequency table
            token_freqs = [
                (token, s2_address_token_counts[token])
                for token in tokens
                if token in s2_address_token_counts
                and s2_address_token_counts[token] <= 500
            ]

            # Rarest tokens first
            token_freqs.sort(key=lambda x: x[1])

            for token, _ in token_freqs[:k]:
                candidates.update(
                    s2_address_token_index.get(token, [])
                )

        recovered = gt_ids & candidates

        recovered_pairs += len(recovered)
        candidate_counts.append(len(candidates))

        if len(gt_ids) == len(recovered):
            complete_recall_count += 1

    rarest_address_block_s2.append({
        "k_rarest_tokens": k,
        "pair_recall": recovered_pairs / total_pairs,
        "mean_candidates": np.mean(candidate_counts),
        "median_candidates": np.median(candidate_counts),
        "p90_candidates": np.percentile(candidate_counts, 90),
        "p95_candidates": np.percentile(candidate_counts, 95),
        "p99_candidates": np.percentile(candidate_counts, 99),
        "max_candidates": np.max(candidate_counts),
        "complete_s1_recall": (
            complete_recall_count / len(blocking_sample)
        ),
    })

rarest_address_block_s2 = pd.DataFrame(
    rarest_address_block_s2
)

display(rarest_address_block_s2)

,k_rarest_tokens,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,1,0.314225,88.0828,24.0,285.0,375.05,472.01,500,0.0960
1,2,0.330694,159.4786,92.0,427.1,496.00,715.01,969,0.0991
2,3,0.331590,187.0287,122.0,474.0,578.00,799.05,1247,0.0994


In [ ]:
print("S2 name token index:", len(s2_token_index))
print("S3 name token index:", len(s3_token_index))

print("S2 name frequencies:", len(s2_token_frequency))
print("S3 name frequencies:", len(s3_name_token_counts))

print("S2 address index:", len(s2_address_token_index))
print("S3 address index:", len(s3_address_token_index))


S2 name token index: 809965
S3 name token index: 862459
S2 name frequencies: 810514
S3 name frequencies: 863014
S2 address index: 607152
S3 address index: 579246


In [ ]:
COMBINED_TOKEN_THRESHOLD = 500
TOP_K = 2

combined_rare_token_summary = []

for row in blocking_sample[
    [
        "source1_entity_id",
        "normalized_name",
        "normalized_address",
        "matched_entity_ids",
    ]
].itertuples(index=False):

    gt_ids = set(parse_gt_ids(row.matched_entity_ids))

    # ---------------------------------------------------------
    # Exact blocks
    # ---------------------------------------------------------
    name_candidates = (
        set(s1_name_candidates_s2.get(row.source1_entity_id, []))
        | set(s1_name_candidates_s3.get(row.source1_entity_id, []))
    )

    address_candidates = (
        set(s1_address_candidates_s2.get(row.source1_entity_id, []))
        | set(s1_address_candidates_s3.get(row.source1_entity_id, []))
    )

    # ---------------------------------------------------------
    # Rare name tokens
    # ---------------------------------------------------------
    rare_name_s2 = []
    rare_name_s3 = []

    if row.normalized_name:
        name_tokens = set(row.normalized_name.split())

        for token in name_tokens:
            freq_s2 = s2_token_frequency.get(token)

            if freq_s2 is not None and freq_s2 <= COMBINED_TOKEN_THRESHOLD:
                rare_name_s2.append((token, freq_s2))

            freq_s3 = s3_name_token_counts.get(token)

            if freq_s3 is not None and freq_s3 <= COMBINED_TOKEN_THRESHOLD:
                rare_name_s3.append((token, freq_s3))

    rare_name_s2.sort(key=lambda x: x[1])
    rare_name_s3.sort(key=lambda x: x[1])

    name_token_candidates = set()

    for token, _ in rare_name_s2[:TOP_K]:
        name_token_candidates.update(
            s2_token_index.get(token, [])
        )

    for token, _ in rare_name_s3[:TOP_K]:
        name_token_candidates.update(
            s3_token_index.get(token, [])
        )

    # ---------------------------------------------------------
    # Rare address tokens
    # ---------------------------------------------------------
    rare_address_s2 = []
    rare_address_s3 = []

    if row.normalized_address:
        address_tokens = set(row.normalized_address.split())

        for token in address_tokens:
            freq_s2 = s2_address_token_counts.get(token)

            if freq_s2 is not None and freq_s2 <= COMBINED_TOKEN_THRESHOLD:
                rare_address_s2.append((token, freq_s2))

            freq_s3 = s3_address_token_counts.get(token)

            if freq_s3 is not None and freq_s3 <= COMBINED_TOKEN_THRESHOLD:
                rare_address_s3.append((token, freq_s3))

    rare_address_s2.sort(key=lambda x: x[1])
    rare_address_s3.sort(key=lambda x: x[1])

    address_token_candidates = set()

    for token, _ in rare_address_s2[:TOP_K]:
        address_token_candidates.update(
            s2_address_token_index.get(token, [])
        )

    for token, _ in rare_address_s3[:TOP_K]:
        address_token_candidates.update(
            s3_address_token_index.get(token, [])
        )

    # ---------------------------------------------------------
    # Final candidate union
    # ---------------------------------------------------------
    combined_candidates = (
        name_candidates
        | address_candidates
        | name_token_candidates
        | address_token_candidates
    )

    recovered = gt_ids & combined_candidates

    candidate_count = len(combined_candidates)

    combined_rare_token_summary.append({
        "candidate_count": candidate_count,
        "gt_count": len(gt_ids),
        "recovered_count": len(recovered),
        "complete_recall": len(gt_ids) == len(recovered),
    })


combined_rare_token_summary = pd.DataFrame(
    combined_rare_token_summary
)

total_pairs = combined_rare_token_summary["gt_count"].sum()
recovered_pairs = combined_rare_token_summary["recovered_count"].sum()

summary = pd.DataFrame([{
    "threshold": COMBINED_TOKEN_THRESHOLD,
    "top_k_per_field": TOP_K,
    "pair_recall": recovered_pairs / total_pairs,
    "mean_candidates": combined_rare_token_summary["candidate_count"].mean(),
    "median_candidates": combined_rare_token_summary["candidate_count"].median(),
    "p90_candidates": np.percentile(
        combined_rare_token_summary["candidate_count"], 90
    ),
    "p95_candidates": np.percentile(
        combined_rare_token_summary["candidate_count"], 95
    ),
    "p99_candidates": np.percentile(
        combined_rare_token_summary["candidate_count"], 99
    ),
    "max_candidates": combined_rare_token_summary["candidate_count"].max(),
    "complete_s1_recall": (
        combined_rare_token_summary["complete_recall"].mean()
    ),
}])

display(summary)

,threshold,top_k_per_field,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,500,2,0.847034,454.3949,359.0,1022.2,1255.0,1676.0,2881,0.7254


In [ ]:
COMBINED_THRESHOLDS = [50, 100, 250, 500]
TOP_K = 2

combined_rare_token_results = []

for threshold in COMBINED_THRESHOLDS:

    total_pairs = 0
    recovered_pairs = 0
    candidate_counts = []
    complete_recall_count = 0

    for row in blocking_sample[
        [
            "source1_entity_id",
            "normalized_name",
            "normalized_address",
            "matched_entity_ids",
        ]
    ].itertuples(index=False):

        gt_ids = set(parse_gt_ids(row.matched_entity_ids))
        total_pairs += len(gt_ids)

        # --------------------------------------------------
        # Exact blocks
        # --------------------------------------------------

        candidates = (
            set(
                s1_name_candidates_s2.get(
                    row.source1_entity_id, []
                )
            )
            | set(
                s1_name_candidates_s3.get(
                    row.source1_entity_id, []
                )
            )
            | set(
                s1_address_candidates_s2.get(
                    row.source1_entity_id, []
                )
            )
            | set(
                s1_address_candidates_s3.get(
                    row.source1_entity_id, []
                )
            )
        )

        # --------------------------------------------------
        # Name: 2 rarest eligible tokens
        # --------------------------------------------------

        if row.normalized_name:

            name_tokens = set(row.normalized_name.split())

            s2_name_tokens = [
                (token, s2_token_frequency[token])
                for token in name_tokens
                if token in s2_token_frequency
                and s2_token_frequency[token] <= threshold
            ]

            s3_name_tokens = [
                (token, s3_name_token_counts[token])
                for token in name_tokens
                if token in s3_name_token_counts
                and s3_name_token_counts[token] <= threshold
            ]

            s2_name_tokens.sort(key=lambda x: x[1])
            s3_name_tokens.sort(key=lambda x: x[1])

            for token, _ in s2_name_tokens[:TOP_K]:
                candidates.update(
                    s2_token_index.get(token, [])
                )

            for token, _ in s3_name_tokens[:TOP_K]:
                candidates.update(
                    s3_token_index.get(token, [])
                )

        # --------------------------------------------------
        # Address: 2 rarest eligible tokens
        # --------------------------------------------------

        if row.normalized_address:

            address_tokens = set(
                row.normalized_address.split()
            )

            s2_address_tokens = [
                (token, s2_address_token_counts[token])
                for token in address_tokens
                if token in s2_address_token_counts
                and s2_address_token_counts[token] <= threshold
            ]

            s3_address_tokens = [
                (token, s3_address_token_counts[token])
                for token in address_tokens
                if token in s3_address_token_counts
                and s3_address_token_counts[token] <= threshold
            ]

            s2_address_tokens.sort(key=lambda x: x[1])
            s3_address_tokens.sort(key=lambda x: x[1])

            for token, _ in s2_address_tokens[:TOP_K]:
                candidates.update(
                    s2_address_token_index.get(token, [])
                )

            for token, _ in s3_address_tokens[:TOP_K]:
                candidates.update(
                    s3_address_token_index.get(token, [])
                )

        # --------------------------------------------------
        # Evaluate
        # --------------------------------------------------

        recovered = gt_ids & candidates

        recovered_pairs += len(recovered)

        candidate_counts.append(len(candidates))

        if len(gt_ids) == len(recovered):
            complete_recall_count += 1

    combined_rare_token_results.append({
        "threshold": threshold,
        "top_k": TOP_K,
        "pair_recall": recovered_pairs / total_pairs,
        "mean_candidates": np.mean(candidate_counts),
        "median_candidates": np.median(candidate_counts),
        "p90_candidates": np.percentile(
            candidate_counts, 90
        ),
        "p95_candidates": np.percentile(
            candidate_counts, 95
        ),
        "p99_candidates": np.percentile(
            candidate_counts, 99
        ),
        "max_candidates": np.max(candidate_counts),
        "complete_s1_recall": (
            complete_recall_count / len(blocking_sample)
        ),
    })

combined_rare_token_results = pd.DataFrame(
    combined_rare_token_results
)

display(combined_rare_token_results)

,threshold,top_k,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,50,2,0.584414,32.2044,13.0,87.0,110.0,196.02,460,0.3983
1,100,2,0.680025,76.9911,49.0,191.0,240.0,333.00,566,0.5055
2,250,2,0.786558,217.3052,166.0,509.0,631.0,846.00,1260,0.6419
3,500,2,0.847034,454.3949,359.0,1022.2,1255.0,1676.00,2881,0.7254


In [ ]:
CROSS_THRESHOLD = 500
CROSS_TOP_K = 2

cross_field_results = []

for row in blocking_sample[
    [
        "source1_entity_id",
        "normalized_name",
        "normalized_address",
        "matched_entity_ids",
    ]
].itertuples(index=False):

    gt_ids = set(parse_gt_ids(row.matched_entity_ids))

    # --------------------------------------------------
    # S2 rare name candidates
    # --------------------------------------------------

    s2_name_candidates = set()

    if row.normalized_name:
        name_tokens = set(row.normalized_name.split())

        eligible = [
            (token, s2_token_frequency[token])
            for token in name_tokens
            if token in s2_token_frequency
            and s2_token_frequency[token] <= CROSS_THRESHOLD
        ]

        eligible.sort(key=lambda x: x[1])

        for token, _ in eligible[:CROSS_TOP_K]:
            s2_name_candidates.update(
                s2_token_index.get(token, [])
            )

    # --------------------------------------------------
    # S3 rare name candidates
    # --------------------------------------------------

    s3_name_candidates = set()

    if row.normalized_name:
        name_tokens = set(row.normalized_name.split())

        eligible = [
            (token, s3_name_token_counts[token])
            for token in name_tokens
            if token in s3_name_token_counts
            and s3_name_token_counts[token] <= CROSS_THRESHOLD
        ]

        eligible.sort(key=lambda x: x[1])

        for token, _ in eligible[:CROSS_TOP_K]:
            s3_name_candidates.update(
                s3_token_index.get(token, [])
            )

    # --------------------------------------------------
    # S2 rare address candidates
    # --------------------------------------------------

    s2_address_candidates = set()

    if row.normalized_address:
        address_tokens = set(row.normalized_address.split())

        eligible = [
            (token, s2_address_token_counts[token])
            for token in address_tokens
            if token in s2_address_token_counts
            and s2_address_token_counts[token] <= CROSS_THRESHOLD
        ]

        eligible.sort(key=lambda x: x[1])

        for token, _ in eligible[:CROSS_TOP_K]:
            s2_address_candidates.update(
                s2_address_token_index.get(token, [])
            )

    # --------------------------------------------------
    # S3 rare address candidates
    # --------------------------------------------------

    s3_address_candidates = set()

    if row.normalized_address:
        address_tokens = set(row.normalized_address.split())

        eligible = [
            (token, s3_address_token_counts[token])
            for token in address_tokens
            if token in s3_address_token_counts
            and s3_address_token_counts[token] <= CROSS_THRESHOLD
        ]

        eligible.sort(key=lambda x: x[1])

        for token, _ in eligible[:CROSS_TOP_K]:
            s3_address_candidates.update(
                s3_address_token_index.get(token, [])
            )

    # --------------------------------------------------
    # Cross-field AND
    #
    # A candidate must share:
    #   name token AND address token
    # --------------------------------------------------

    s2_cross_candidates = (
        s2_name_candidates & s2_address_candidates
    )

    s3_cross_candidates = (
        s3_name_candidates & s3_address_candidates
    )

    candidates = (
        s2_cross_candidates
        | s3_cross_candidates
    )

    recovered = gt_ids & candidates

    cross_field_results.append({
        "candidate_count": len(candidates),
        "gt_count": len(gt_ids),
        "recovered_count": len(recovered),
        "complete_recall": len(gt_ids) == len(recovered),
    })


cross_field_results = pd.DataFrame(cross_field_results)

total_pairs = cross_field_results["gt_count"].sum()
recovered_pairs = cross_field_results["recovered_count"].sum()

cross_field_summary = pd.DataFrame([{
    "threshold": CROSS_THRESHOLD,
    "top_k": CROSS_TOP_K,
    "pair_recall": recovered_pairs / total_pairs,
    "mean_candidates": cross_field_results["candidate_count"].mean(),
    "median_candidates": cross_field_results["candidate_count"].median(),
    "p90_candidates": np.percentile(
        cross_field_results["candidate_count"], 90
    ),
    "p95_candidates": np.percentile(
        cross_field_results["candidate_count"], 95
    ),
    "p99_candidates": np.percentile(
        cross_field_results["candidate_count"], 99
    ),
    "max_candidates": cross_field_results["candidate_count"].max(),
    "complete_s1_recall": (
        cross_field_results["complete_recall"].mean()
    ),
}])

display(cross_field_summary)

,threshold,top_k,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,500,2,0.281083,1.3203,0.0,4.0,5.0,8.0,51,0.2041


In [ ]:
CROSS_THRESHOLD = 500
CROSS_K_VALUES = [1, 2, 3]

cross_field_k_results = []

for cross_k in CROSS_K_VALUES:

    total_pairs = 0
    recovered_pairs = 0
    candidate_counts = []
    complete_recall_count = 0

    for row in blocking_sample[
        [
            "source1_entity_id",
            "normalized_name",
            "normalized_address",
            "matched_entity_ids",
        ]
    ].itertuples(index=False):

        gt_ids = set(parse_gt_ids(row.matched_entity_ids))
        total_pairs += len(gt_ids)

        # --------------------------------------------------
        # S2 name candidates
        # --------------------------------------------------

        s2_name_candidates = set()

        if row.normalized_name:
            tokens = set(row.normalized_name.split())

            eligible = [
                (token, s2_token_frequency[token])
                for token in tokens
                if token in s2_token_frequency
                and s2_token_frequency[token] <= CROSS_THRESHOLD
            ]

            eligible.sort(key=lambda x: x[1])

            for token, _ in eligible[:cross_k]:
                s2_name_candidates.update(
                    s2_token_index.get(token, [])
                )

        # --------------------------------------------------
        # S3 name candidates
        # --------------------------------------------------

        s3_name_candidates = set()

        if row.normalized_name:
            tokens = set(row.normalized_name.split())

            eligible = [
                (token, s3_name_token_counts[token])
                for token in tokens
                if token in s3_name_token_counts
                and s3_name_token_counts[token] <= CROSS_THRESHOLD
            ]

            eligible.sort(key=lambda x: x[1])

            for token, _ in eligible[:cross_k]:
                s3_name_candidates.update(
                    s3_token_index.get(token, [])
                )

        # --------------------------------------------------
        # S2 address candidates
        # --------------------------------------------------

        s2_address_candidates = set()

        if row.normalized_address:
            tokens = set(row.normalized_address.split())

            eligible = [
                (token, s2_address_token_counts[token])
                for token in tokens
                if token in s2_address_token_counts
                and s2_address_token_counts[token] <= CROSS_THRESHOLD
            ]

            eligible.sort(key=lambda x: x[1])

            for token, _ in eligible[:cross_k]:
                s2_address_candidates.update(
                    s2_address_token_index.get(token, [])
                )

        # --------------------------------------------------
        # S3 address candidates
        # --------------------------------------------------

        s3_address_candidates = set()

        if row.normalized_address:
            tokens = set(row.normalized_address.split())

            eligible = [
                (token, s3_address_token_counts[token])
                for token in tokens
                if token in s3_address_token_counts
                and s3_address_token_counts[token] <= CROSS_THRESHOLD
            ]

            eligible.sort(key=lambda x: x[1])

            for token, _ in eligible[:cross_k]:
                s3_address_candidates.update(
                    s3_address_token_index.get(token, [])
                )

        # --------------------------------------------------
        # AND within each target source
        # --------------------------------------------------

        candidates = (
            s2_name_candidates & s2_address_candidates
        ) | (
            s3_name_candidates & s3_address_candidates
        )

        recovered = gt_ids & candidates

        recovered_pairs += len(recovered)

        candidate_counts.append(len(candidates))

        if len(gt_ids) == len(recovered):
            complete_recall_count += 1

    cross_field_k_results.append({
        "threshold": CROSS_THRESHOLD,
        "top_k": cross_k,
        "pair_recall": recovered_pairs / total_pairs,
        "mean_candidates": np.mean(candidate_counts),
        "median_candidates": np.median(candidate_counts),
        "p90_candidates": np.percentile(
            candidate_counts, 90
        ),
        "p95_candidates": np.percentile(
            candidate_counts, 95
        ),
        "p99_candidates": np.percentile(
            candidate_counts, 99
        ),
        "max_candidates": np.max(candidate_counts),
        "complete_s1_recall": (
            complete_recall_count / len(blocking_sample)
        ),
    })

cross_field_k_results = pd.DataFrame(
    cross_field_k_results
)

display(cross_field_k_results)


,threshold,top_k,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,500,1,0.262764,1.1689,0.0,4.0,5.0,7.0,40,0.1812
1,500,2,0.281083,1.3203,0.0,4.0,5.0,8.0,51,0.2041
2,500,3,0.283019,1.3640,0.0,4.0,5.0,8.0,60,0.2070


In [ ]:
def generate_block_candidates(
    s1_row,
    threshold=500,
    top_k=2,
):
    """
    Generate S2/S3 candidate entity IDs for one S1 record.

    Blocking rules:
      1. Exact normalized name
      2. Exact normalized address
      3. Top-k rarest eligible name tokens
      4. Top-k rarest eligible address tokens

    Returns
    -------
    set
        Candidate entity IDs from S2 and S3.
    """

    s1_id = s1_row["source1_entity_id"]

    candidates = set()

    # =====================================================
    # Exact normalized name
    # =====================================================

    candidates.update(
        s1_name_candidates_s2.get(s1_id, [])
    )

    candidates.update(
        s1_name_candidates_s3.get(s1_id, [])
    )

    # =====================================================
    # Exact normalized address
    # =====================================================

    candidates.update(
        s1_address_candidates_s2.get(s1_id, [])
    )

    candidates.update(
        s1_address_candidates_s3.get(s1_id, [])
    )

    # =====================================================
    # Rare name tokens
    # =====================================================

    normalized_name = s1_row["normalized_name"]

    if normalized_name:

        name_tokens = set(
            normalized_name.split()
        )

        # ---------- S2 ----------

        s2_tokens = [
            (token, s2_token_frequency[token])
            for token in name_tokens
            if token in s2_token_frequency
            and s2_token_frequency[token] <= threshold
        ]

        s2_tokens.sort(
            key=lambda x: x[1]
        )

        for token, _ in s2_tokens[:top_k]:

            candidates.update(
                s2_token_index.get(token, [])
            )

        # ---------- S3 ----------

        s3_tokens = [
            (token, s3_name_token_counts[token])
            for token in name_tokens
            if token in s3_name_token_counts
            and s3_name_token_counts[token] <= threshold
        ]

        s3_tokens.sort(
            key=lambda x: x[1]
        )

        for token, _ in s3_tokens[:top_k]:

            candidates.update(
                s3_token_index.get(token, [])
            )

    # =====================================================
    # Rare address tokens
    # =====================================================

    normalized_address = s1_row["normalized_address"]

    if normalized_address:

        address_tokens = set(
            normalized_address.split()
        )

        # ---------- S2 ----------

        s2_tokens = [
            (token, s2_address_token_counts[token])
            for token in address_tokens
            if token in s2_address_token_counts
            and s2_address_token_counts[token] <= threshold
        ]

        s2_tokens.sort(
            key=lambda x: x[1]
        )

        for token, _ in s2_tokens[:top_k]:

            candidates.update(
                s2_address_token_index.get(token, [])
            )

        # ---------- S3 ----------

        s3_tokens = [
            (token, s3_address_token_counts[token])
            for token in address_tokens
            if token in s3_address_token_counts
            and s3_address_token_counts[token] <= threshold
        ]

        s3_tokens.sort(
            key=lambda x: x[1]
        )

        for token, _ in s3_tokens[:top_k]:

            candidates.update(
                s3_address_token_index.get(token, [])
            )

    return candidates


print("Blocking function defined successfully.")

Blocking function defined successfully.


In [ ]:
# Reproduce the known 10k blocking benchmark
# Expected pair recall: ~0.847034
# Expected complete S1 recall: ~0.7254

total_pairs = 0
recovered_pairs = 0

candidate_counts = []
complete_recall_count = 0

for _, row in blocking_sample[
    [
        "source1_entity_id",
        "normalized_name",
        "normalized_address",
        "matched_entity_ids",
    ]
].iterrows():

    gt_ids = set(parse_gt_ids(row["matched_entity_ids"]))

    candidates = generate_block_candidates(
        row,
        threshold=500,
        top_k=2,
    )

    recovered = gt_ids & candidates

    total_pairs += len(gt_ids)
    recovered_pairs += len(recovered)

    candidate_counts.append(len(candidates))

    if len(gt_ids) == len(recovered):
        complete_recall_count += 1


sanity_result = pd.DataFrame([{
    "threshold": 500,
    "top_k": 2,
    "pair_recall": recovered_pairs / total_pairs,
    "mean_candidates": np.mean(candidate_counts),
    "median_candidates": np.median(candidate_counts),
    "p90_candidates": np.percentile(candidate_counts, 90),
    "p95_candidates": np.percentile(candidate_counts, 95),
    "p99_candidates": np.percentile(candidate_counts, 99),
    "max_candidates": np.max(candidate_counts),
    "complete_s1_recall": (
        complete_recall_count / len(blocking_sample)
    ),
}])

display(sanity_result)

,threshold,top_k,pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,complete_s1_recall
0,500,2,0.847034,454.3949,359.0,1022.2,1255.0,1676.0,2881,0.7254


In [ ]:
# Load validation S1 IDs
valid_s1_ids = pd.read_csv(
    VALID_IDS_PATH,
    sep="\t",
    dtype={"source1_entity_id": str},
)

print("Validation IDs:", len(valid_s1_ids))
print("Unique IDs:", valid_s1_ids["source1_entity_id"].nunique())

# Attach the actual S1 records
validation_s1 = valid_s1_ids.merge(
    s1,
    left_on="source1_entity_id",
    right_on="entity_id",
    how="left",
    validate="one_to_one",
)

# Keep the ID name consistent
validation_s1 = validation_s1.drop(
    columns=["entity_id"]
)

# Apply the exact same normalization used in blocking
validation_s1["normalized_name"] = (
    validation_s1["business_name"].map(normalize_text)
)

validation_s1["normalized_address"] = (
    validation_s1["business_address"].map(normalize_text)
)

print("\nValidation S1 shape:", validation_s1.shape)

print(
    "Missing business names:",
    validation_s1["business_name"].isna().sum()
)

print(
    "Missing business addresses:",
    validation_s1["business_address"].isna().sum()
)

display(validation_s1.head())

Validation IDs: 441365
Unique IDs: 441365

Validation S1 shape: (441365, 6)
Missing business names: 0
Missing business addresses: 0


,source1_entity_id,business_name,business_address,country,normalized_name,normalized_address
0,S1-392794459,Av Meadows Private Limited,"1-C, Officers Club Road, Kadambad, Nagapattina...",India,av meadows private limited,1 c officers club road kadambad nagapattinam 6...
1,S1-169028455,Rocky Guild,"7482 Kowaliga Road, Eclectic, AL",US,rocky guild,7482 kowaliga road eclectic al
2,S1-231659389,Lantern,"1122 Kildaire Farm Road, Cary, NC",US,lantern,1122 kildaire farm road cary nc
3,S1-873273895,Better Defense Works Corp,"56 Persimmon Lane, Chester, WV",US,better defense works corp,56 persimmon lane chester wv
4,S1-198141680,OV Ventures Pvt Ltd,"D-77 Defence Colony, New Delhi, South Delhi, D...",India,ov ventures pvt ltd,d 77 defence colony new delhi south delhi delhi


In [ ]:
# Load validation ground truth
validation_gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
)

validation_gt["matched_entity_ids"] = (
    validation_gt["matched_entity_ids"]
    .fillna("")
)

# Keep only validation S1 IDs
validation_eval = validation_s1.merge(
    validation_gt[
        ["source1_entity_id", "matched_entity_ids"]
    ],
    on="source1_entity_id",
    how="left",
    validate="one_to_one",
)

print("Validation evaluation rows:", len(validation_eval))

# ---------------------------------------------------------
# Evaluate blocking on the full validation set
# ---------------------------------------------------------

total_pairs = 0
recovered_pairs = 0

candidate_counts = []

complete_recall_count = 0
zero_candidate_count = 0

# Source-specific recall
total_s2_pairs = 0
recovered_s2_pairs = 0

total_s3_pairs = 0
recovered_s3_pairs = 0


for _, row in validation_eval[
    [
        "source1_entity_id",
        "normalized_name",
        "normalized_address",
        "matched_entity_ids",
    ]
].iterrows():

    gt_ids = set(
        parse_gt_ids(row["matched_entity_ids"])
    )

    candidates = generate_block_candidates(
        row,
        threshold=500,
        top_k=2,
    )

    recovered = gt_ids & candidates

    # -----------------------------------------------------
    # Overall pair recall
    # -----------------------------------------------------

    total_pairs += len(gt_ids)
    recovered_pairs += len(recovered)

    candidate_counts.append(len(candidates))

    # -----------------------------------------------------
    # Complete S1 recall
    # -----------------------------------------------------

    if len(gt_ids) == len(recovered):
        complete_recall_count += 1

    # -----------------------------------------------------
    # Zero-candidate diagnostic
    # -----------------------------------------------------

    if len(candidates) == 0:
        zero_candidate_count += 1

    # -----------------------------------------------------
    # S2-specific recall
    # -----------------------------------------------------

    gt_s2 = {
        entity_id
        for entity_id in gt_ids
        if entity_id.startswith("S2-")
    }

    recovered_s2 = gt_s2 & candidates

    total_s2_pairs += len(gt_s2)
    recovered_s2_pairs += len(recovered_s2)

    # -----------------------------------------------------
    # S3-specific recall
    # -----------------------------------------------------

    gt_s3 = {
        entity_id
        for entity_id in gt_ids
        if entity_id.startswith("S3-")
    }

    recovered_s3 = gt_s3 & candidates

    total_s3_pairs += len(gt_s3)
    recovered_s3_pairs += len(recovered_s3)


validation_blocking_summary = pd.DataFrame([{
    "threshold": 500,
    "top_k": 2,

    "total_s1": len(validation_eval),

    "total_gt_pairs": total_pairs,

    "pair_recall": (
        recovered_pairs / total_pairs
        if total_pairs
        else 0
    ),

    "s2_pair_recall": (
        recovered_s2_pairs / total_s2_pairs
        if total_s2_pairs
        else 0
    ),

    "s3_pair_recall": (
        recovered_s3_pairs / total_s3_pairs
        if total_s3_pairs
        else 0
    ),

    "mean_candidates": np.mean(candidate_counts),

    "median_candidates": np.median(candidate_counts),

    "p90_candidates": np.percentile(
        candidate_counts, 90
    ),

    "p95_candidates": np.percentile(
        candidate_counts, 95
    ),

    "p99_candidates": np.percentile(
        candidate_counts, 99
    ),

    "max_candidates": np.max(candidate_counts),

    "zero_candidate_s1": zero_candidate_count,

    "zero_candidate_rate": (
        zero_candidate_count / len(validation_eval)
    ),

    "complete_s1_recall": (
        complete_recall_count / len(validation_eval)
    ),
}])

display(validation_blocking_summary)

Validation evaluation rows: 441365


,threshold,top_k,total_s1,total_gt_pairs,pair_recall,s2_pair_recall,s3_pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,zero_candidate_s1,zero_candidate_rate,complete_s1_recall
0,500,2,441365,1527679,0.804407,0.812679,0.796649,453.956999,364.0,1028.0,1253.0,1683.0,3083,47576,0.107793,0.698592


In [ ]:
# Test the full blocker with top_k=3
# threshold remains 500

total_pairs = 0
recovered_pairs = 0

candidate_counts = []

complete_recall_count = 0
zero_candidate_count = 0

total_s2_pairs = 0
recovered_s2_pairs = 0

total_s3_pairs = 0
recovered_s3_pairs = 0


for _, row in validation_eval[
    [
        "source1_entity_id",
        "normalized_name",
        "normalized_address",
        "matched_entity_ids",
    ]
].iterrows():

    gt_ids = set(
        parse_gt_ids(row["matched_entity_ids"])
    )

    candidates = generate_block_candidates(
        row,
        threshold=500,
        top_k=3,
    )

    recovered = gt_ids & candidates

    total_pairs += len(gt_ids)
    recovered_pairs += len(recovered)

    candidate_counts.append(len(candidates))

    if len(gt_ids) == len(recovered):
        complete_recall_count += 1

    if len(candidates) == 0:
        zero_candidate_count += 1

    gt_s2 = {
        entity_id
        for entity_id in gt_ids
        if entity_id.startswith("S2-")
    }

    recovered_s2 = gt_s2 & candidates

    total_s2_pairs += len(gt_s2)
    recovered_s2_pairs += len(recovered_s2)

    gt_s3 = {
        entity_id
        for entity_id in gt_ids
        if entity_id.startswith("S3-")
    }

    recovered_s3 = gt_s3 & candidates

    total_s3_pairs += len(gt_s3)
    recovered_s3_pairs += len(recovered_s3)


validation_blocking_k3_summary = pd.DataFrame([{
    "threshold": 500,
    "top_k": 3,
    "total_s1": len(validation_eval),
    "total_gt_pairs": total_pairs,

    "pair_recall": (
        recovered_pairs / total_pairs
        if total_pairs else 0
    ),

    "s2_pair_recall": (
        recovered_s2_pairs / total_s2_pairs
        if total_s2_pairs else 0
    ),

    "s3_pair_recall": (
        recovered_s3_pairs / total_s3_pairs
        if total_s3_pairs else 0
    ),

    "mean_candidates": np.mean(candidate_counts),
    "median_candidates": np.median(candidate_counts),

    "p90_candidates": np.percentile(
        candidate_counts, 90
    ),

    "p95_candidates": np.percentile(
        candidate_counts, 95
    ),

    "p99_candidates": np.percentile(
        candidate_counts, 99
    ),

    "max_candidates": np.max(candidate_counts),

    "zero_candidate_s1": zero_candidate_count,

    "zero_candidate_rate": (
        zero_candidate_count / len(validation_eval)
    ),

    "complete_s1_recall": (
        complete_recall_count / len(validation_eval)
    ),
}])

display(validation_blocking_k3_summary)

,threshold,top_k,total_s1,total_gt_pairs,pair_recall,s2_pair_recall,s3_pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,zero_candidate_s1,zero_candidate_rate,complete_s1_recall
0,500,3,441365,1527679,0.806699,0.81452,0.799365,512.92143,429.0,1146.0,1392.0,1869.0,3750,47576,0.107793,0.703388


In [ ]:
# ---------------------------------------------------------
# Analyze WHY validation ground-truth pairs are missed
# ---------------------------------------------------------

missed_rows = []

for _, row in validation_eval[
    [
        "source1_entity_id",
        "normalized_name",
        "normalized_address",
        "matched_entity_ids",
    ]
].iterrows():

    gt_ids = set(
        parse_gt_ids(row["matched_entity_ids"])
    )

    candidates = generate_block_candidates(
        row,
        threshold=500,
        top_k=2,
    )

    missed_ids = gt_ids - candidates

    if missed_ids:
        missed_rows.append({
            "source1_entity_id": row["source1_entity_id"],
            "normalized_name": row["normalized_name"],
            "normalized_address": row["normalized_address"],
            "matched_entity_ids": row["matched_entity_ids"],
            "missed_count": len(missed_ids),
            "missed_entity_ids": ",".join(sorted(missed_ids)),
        })

missed_validation = pd.DataFrame(missed_rows)

print("S1 records with at least one missed GT match:",
      len(missed_validation))

print(
    "Total missed GT pairs:",
    missed_validation["missed_count"].sum()
)

display(
    missed_validation[
        ["missed_count"]
    ].describe()
)

S1 records with at least one missed GT match: 133031
Total missed GT pairs: 298804


,missed_count
count,133031.000000
mean,2.246123
std,1.532345
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,11.000000


In [ ]:
# ---------------------------------------------------------
# Inspect actual missed validation pairs
# ---------------------------------------------------------

# Take 30 missed S1 records
missed_sample = missed_validation.sample(
    n=min(30, len(missed_validation)),
    random_state=42,
)

# Get all missed target IDs
missed_target_ids = set()

for ids in missed_sample["missed_entity_ids"]:
    missed_target_ids.update(ids.split(","))

missed_target_ids = list(missed_target_ids)

print("Missed target IDs to inspect:", len(missed_target_ids))


# ---------------------------------------------------------
# Retrieve target records from S2/S3
# ---------------------------------------------------------

s2_missed = s2[
    s2["entity_id"].isin(missed_target_ids)
].copy()

s3_missed = s3[
    s3["entity_id"].isin(missed_target_ids)
].copy()

targets_missed = pd.concat(
    [
        s2_missed.assign(target_source="S2"),
        s3_missed.assign(target_source="S3"),
    ],
    ignore_index=True,
)

targets_missed = targets_missed.rename(
    columns={
        "entity_id": "target_entity_id",
        "business_name": "target_business_name",
        "business_address": "target_business_address",
        "country": "target_country",
    }
)


# ---------------------------------------------------------
# Expand S1 -> individual missed target pairs
# ---------------------------------------------------------

missed_pairs = []

for _, row in missed_sample.iterrows():

    s1_id = row["source1_entity_id"]

    for target_id in row["missed_entity_ids"].split(","):

        missed_pairs.append({
            "source1_entity_id": s1_id,
            "target_entity_id": target_id,
            "s1_business_name": row["normalized_name"],
            "s1_business_address": row["normalized_address"],
        })


missed_pairs = pd.DataFrame(missed_pairs)


# ---------------------------------------------------------
# Attach target information
# ---------------------------------------------------------

missed_pairs = missed_pairs.merge(
    targets_missed[
        [
            "target_entity_id",
            "target_source",
            "target_business_name",
            "target_business_address",
            "target_country",
        ]
    ],
    on="target_entity_id",
    how="left",
)


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

display(
    missed_pairs[
        [
            "source1_entity_id",
            "target_entity_id",
            "target_source",
            "s1_business_name",
            "target_business_name",
            "s1_business_address",
            "target_business_address",
            "target_country",
        ]
    ]
)

Missed target IDs to inspect: 51


,source1_entity_id,target_entity_id,target_source,s1_business_name,target_business_name,s1_business_address,target_business_address,target_country
0,S1-469874477,S2-81284524,S2,all state inc,All Shttaee Inc,2011 homewood avenue baltimore md,"BALTIMORE, MD, 002011 HOMEWOOD AVE",US
1,S1-371853970,S2-365150861,S2,m c x dome inc,Inc M C & X Dóme,33 evergreen court orem ut,"#33 EVERGREEN COURT, OREM, UT",US
2,S1-219036074,S2-900057307,S2,future properties,ఫ్యూచర్ ప్రాపర్టీస్,d no 18 5 43 3 near prabhas college 3rd lane k...,"Andhra Pradesh, D.NO:D/18-5-43/3 NEAR PRABHAS ...",India
3,S1-219036074,S3-841453594,S3,future properties,Future [Próperties],d no 18 5 43 3 near prabhas college 3rd lane k...,"D.no:18-5-43/3 Nera Prabhas College, Srikakula...",India
4,S1-404713471,S3-572088085,S3,bharat global private limited,Bharat Global Private,72 mean road sabeya sabeya ramnagar bettiah we...,"BR, West Champaran, Bettiah, 072",India
5,S1-404713471,S3-733555386,S3,bharat global private limited,Bharat Bharat Global Private,72 mean road sabeya sabeya ramnagar bettiah we...,"072, Bettiah, West Champaran, BR",India
6,S1-404713471,S3-906496391,S3,bharat global private limited,भारत ग्लोबल प्राइवेट लिमिटेड,72 mean road sabeya sabeya ramnagar bettiah we...,"072, Bettiah, West Champaran, बिहार",India
7,S1-621858889,S2-334258632,S2,platinum malt,Platinum Malt,1400 arno street albuquerque nm,NaN,US
8,S1-621858889,S2-599809325,S2,platinum malt,PLATINUM MALT LP,1400 arno street albuquerque nm,NaN,US
9,S1-817516120,S3-538864194,S3,green software limited,green software limited,plot no 33 abcd govt ind estate charkop kandiv...,"#33, Mumbai, Mumbai City, MH",India


In [ ]:
import re
import unicodedata


def normalize_text_v2(value):
    """
    Stronger normalization for entity-resolution blocking.

    Changes:
    - Unicode compatibility normalization
    - Case folding
    - Remove diacritics
    - Punctuation -> spaces
    - Canonicalize common address abbreviations
    - Collapse whitespace
    """

    if pd.isna(value):
        return ""

    text = str(value)

    # -----------------------------------------------------
    # 1. Unicode compatibility normalization
    # -----------------------------------------------------
    text = unicodedata.normalize("NFKC", text)

    # -----------------------------------------------------
    # 2. Remove diacritics
    # -----------------------------------------------------
    text = unicodedata.normalize("NFKD", text)

    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    # -----------------------------------------------------
    # 3. Case normalization
    # -----------------------------------------------------
    text = text.casefold()

    # -----------------------------------------------------
    # 4. Punctuation -> spaces
    # -----------------------------------------------------
    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE,
    )

    # -----------------------------------------------------
    # 5. Address abbreviation normalization
    # -----------------------------------------------------
    replacements = {
        r"\bave\b": "avenue",
        r"\bav\b": "avenue",

        r"\bst\b": "street",

        r"\brd\b": "road",

        r"\bdr\b": "drive",

        r"\bln\b": "lane",

        r"\btrl\b": "trail",

        r"\bhwy\b": "highway",

        r"\bblvd\b": "boulevard",

        r"\bpkwy\b": "parkway",

        r"\bct\b": "court",

        r"\bpl\b": "place",

        r"\bcir\b": "circle",

        r"\bter\b": "terrace",

        r"\bste\b": "suite",

        r"\bapt\b": "apartment",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(
            pattern,
            replacement,
            text,
        )

    # -----------------------------------------------------
    # 6. Collapse whitespace
    # -----------------------------------------------------
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text

In [ ]:
examples = [
    ("Dóme", "dome"),
    ("Próperties", "properties"),
    ("Éxploration", "exploration"),
    ("2011 homewood avenue baltimore md",
     "BALTIMORE, MD, 002011 HOMEWOOD AVE"),
    ("4869 field drive st george city la",
     "4869d Field Drive, St. George City, Louisiana"),
    ("257 huntington trail pittsylvania county va",
     "257 HUNTINGTON TRL, PITTSYLVANIA COUNTY, VA"),
]

for a, b in examples:
    print("=" * 80)
    print("A :", a)
    print("V2:", normalize_text_v2(a))
    print("B :", b)
    print("V2:", normalize_text_v2(b))

A : Dóme
V2: dome
B : dome
V2: dome
A : Próperties
V2: properties
B : properties
V2: properties
A : Éxploration
V2: exploration
B : exploration
V2: exploration
A : 2011 homewood avenue baltimore md
V2: 2011 homewood avenue baltimore md
B : BALTIMORE, MD, 002011 HOMEWOOD AVE
V2: baltimore md 002011 homewood avenue
A : 4869 field drive st george city la
V2: 4869 field drive street george city la
B : 4869d Field Drive, St. George City, Louisiana
V2: 4869d field drive street george city louisiana
A : 257 huntington trail pittsylvania county va
V2: 257 huntington trail pittsylvania county va
B : 257 HUNTINGTON TRL, PITTSYLVANIA COUNTY, VA
V2: 257 huntington trail pittsylvania county va


In [ ]:
def normalize_name_v2(value):
    """
    Stronger normalization for business names.
    """

    if pd.isna(value):
        return ""

    text = str(value)

    # Unicode compatibility normalization
    text = unicodedata.normalize("NFKC", text)

    # Remove diacritics
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    # Case normalization
    text = text.casefold()

    # Punctuation -> spaces
    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE,
    )

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_address_v2(value):
    """
    Stronger normalization for business addresses.
    """

    if pd.isna(value):
        return ""

    text = str(value)

    # Unicode compatibility normalization
    text = unicodedata.normalize("NFKC", text)

    # Remove diacritics
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    # Case normalization
    text = text.casefold()

    # Punctuation -> spaces
    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
        flags=re.UNICODE,
    )

    # Address abbreviations
    replacements = {
        r"\bave\b": "avenue",
        r"\bav\b": "avenue",
        r"\bst\b": "street",
        r"\brd\b": "road",
        r"\bdr\b": "drive",
        r"\bln\b": "lane",
        r"\btrl\b": "trail",
        r"\bhwy\b": "highway",
        r"\bblvd\b": "boulevard",
        r"\bpkwy\b": "parkway",
        r"\bct\b": "court",
        r"\bpl\b": "place",
        r"\bcir\b": "circle",
        r"\bter\b": "terrace",
        r"\bste\b": "suite",
        r"\bapt\b": "apartment",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(
            pattern,
            replacement,
            text,
        )

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
for a, b in examples:
    print("=" * 80)

    print("A NAME:")
    print(normalize_name_v2(a))

    print("A ADDRESS:")
    print(normalize_address_v2(a))

    print("B NAME:")
    print(normalize_name_v2(b))

    print("B ADDRESS:")
    print(normalize_address_v2(b))

A NAME:
dome
A ADDRESS:
dome
B NAME:
dome
B ADDRESS:
dome
A NAME:
properties
A ADDRESS:
properties
B NAME:
properties
B ADDRESS:
properties
A NAME:
exploration
A ADDRESS:
exploration
B NAME:
exploration
B ADDRESS:
exploration
A NAME:
2011 homewood avenue baltimore md
A ADDRESS:
2011 homewood avenue baltimore md
B NAME:
baltimore md 002011 homewood ave
B ADDRESS:
baltimore md 002011 homewood avenue
A NAME:
4869 field drive st george city la
A ADDRESS:
4869 field drive street george city la
B NAME:
4869d field drive st george city louisiana
B ADDRESS:
4869d field drive street george city louisiana
A NAME:
257 huntington trail pittsylvania county va
A ADDRESS:
257 huntington trail pittsylvania county va
B NAME:
257 huntington trl pittsylvania county va
B ADDRESS:
257 huntington trail pittsylvania county va


In [ ]:
# ---------------------------------------------------------
# Test V2 normalization on the inspected missed pairs
# ---------------------------------------------------------

missed_pairs["s1_name_v2"] = missed_pairs["s1_business_name"].map(
    normalize_name_v2
)

missed_pairs["target_name_v2"] = missed_pairs["target_business_name"].map(
    normalize_name_v2
)

missed_pairs["s1_address_v2"] = missed_pairs["s1_business_address"].map(
    normalize_address_v2
)

missed_pairs["target_address_v2"] = missed_pairs["target_business_address"].map(
    normalize_address_v2
)

missed_pairs["name_exact_v2"] = (
    missed_pairs["s1_name_v2"]
    == missed_pairs["target_name_v2"]
)

missed_pairs["address_exact_v2"] = (
    missed_pairs["s1_address_v2"]
    == missed_pairs["target_address_v2"]
)

missed_pairs["either_exact_v2"] = (
    missed_pairs["name_exact_v2"]
    | missed_pairs["address_exact_v2"]
)

print(
    "Missed pairs inspected:",
    len(missed_pairs)
)

print(
    "Recovered by V2 name exact:",
    missed_pairs["name_exact_v2"].sum(),
    f"({missed_pairs['name_exact_v2'].mean():.2%})"
)

print(
    "Recovered by V2 address exact:",
    missed_pairs["address_exact_v2"].sum(),
    f"({missed_pairs['address_exact_v2'].mean():.2%})"
)

print(
    "Recovered by V2 name OR address exact:",
    missed_pairs["either_exact_v2"].sum(),
    f"({missed_pairs['either_exact_v2'].mean():.2%})"
)

display(
    missed_pairs[
        [
            "source1_entity_id",
            "target_entity_id",
            "target_source",
            "s1_business_name",
            "target_business_name",
            "name_exact_v2",
            "s1_business_address",
            "target_business_address",
            "address_exact_v2",
        ]
    ]
)

Missed pairs inspected: 51
Recovered by V2 name exact: 10 (19.61%)
Recovered by V2 address exact: 6 (11.76%)
Recovered by V2 name OR address exact: 16 (31.37%)


,source1_entity_id,target_entity_id,target_source,s1_business_name,target_business_name,name_exact_v2,s1_business_address,target_business_address,address_exact_v2
0,S1-469874477,S2-81284524,S2,all state inc,All Shttaee Inc,False,2011 homewood avenue baltimore md,"BALTIMORE, MD, 002011 HOMEWOOD AVE",False
1,S1-371853970,S2-365150861,S2,m c x dome inc,Inc M C & X Dóme,False,33 evergreen court orem ut,"#33 EVERGREEN COURT, OREM, UT",True
2,S1-219036074,S2-900057307,S2,future properties,ఫ్యూచర్ ప్రాపర్టీస్,False,d no 18 5 43 3 near prabhas college 3rd lane k...,"Andhra Pradesh, D.NO:D/18-5-43/3 NEAR PRABHAS ...",False
3,S1-219036074,S3-841453594,S3,future properties,Future [Próperties],True,d no 18 5 43 3 near prabhas college 3rd lane k...,"D.no:18-5-43/3 Nera Prabhas College, Srikakula...",False
4,S1-404713471,S3-572088085,S3,bharat global private limited,Bharat Global Private,False,72 mean road sabeya sabeya ramnagar bettiah we...,"BR, West Champaran, Bettiah, 072",False
5,S1-404713471,S3-733555386,S3,bharat global private limited,Bharat Bharat Global Private,False,72 mean road sabeya sabeya ramnagar bettiah we...,"072, Bettiah, West Champaran, BR",False
6,S1-404713471,S3-906496391,S3,bharat global private limited,भारत ग्लोबल प्राइवेट लिमिटेड,False,72 mean road sabeya sabeya ramnagar bettiah we...,"072, Bettiah, West Champaran, बिहार",False
7,S1-621858889,S2-334258632,S2,platinum malt,Platinum Malt,True,1400 arno street albuquerque nm,NaN,False
8,S1-621858889,S2-599809325,S2,platinum malt,PLATINUM MALT LP,False,1400 arno street albuquerque nm,NaN,False
9,S1-817516120,S3-538864194,S3,green software limited,green software limited,True,plot no 33 abcd govt ind estate charkop kandiv...,"#33, Mumbai, Mumbai City, MH",False


In [ ]:
# ---------------------------------------------------------
# Create V2 normalized fields
# ---------------------------------------------------------

print("Creating V2 normalized fields...")

# S1
s1["normalized_name_v2"] = s1["business_name"].map(
    normalize_name_v2
)

s1["normalized_address_v2"] = s1["business_address"].map(
    normalize_address_v2
)

# S2
s2["normalized_name_v2"] = s2["business_name"].map(
    normalize_name_v2
)

s2["normalized_address_v2"] = s2["business_address"].map(
    normalize_address_v2
)

# S3
s3["normalized_name_v2"] = s3["business_name"].map(
    normalize_name_v2
)

s3["normalized_address_v2"] = s3["business_address"].map(
    normalize_address_v2
)

print("S1 V2 fields created.")
print("S2 V2 fields created.")
print("S3 V2 fields created.")

print()
print("S1 unique V2 names:",
      s1["normalized_name_v2"].nunique())

print("S2 unique V2 names:",
      s2["normalized_name_v2"].nunique())

print("S3 unique V2 names:",
      s3["normalized_name_v2"].nunique())

print()
print("S2 unique V2 addresses:",
      s2["normalized_address_v2"].nunique())

print("S3 unique V2 addresses:",
      s3["normalized_address_v2"].nunique())

Creating V2 normalized fields...
S1 V2 fields created.
S2 V2 fields created.
S3 V2 fields created.

S1 unique V2 names: 1520684
S2 unique V2 names: 3946649
S3 unique V2 names: 4188983

S2 unique V2 addresses: 4082934
S3 unique V2 addresses: 4431038


In [ ]:
# ---------------------------------------------------------
# Build V2 exact name/address indexes
# ---------------------------------------------------------

from collections import defaultdict

print("Building V2 exact indexes...")

# =========================================================
# NAME INDEXES
# =========================================================

s2_name_index_v2 = (
    s2.groupby("normalized_name_v2")["entity_id"]
    .agg(list)
    .to_dict()
)

s3_name_index_v2 = (
    s3.groupby("normalized_name_v2")["entity_id"]
    .agg(list)
    .to_dict()
)


# =========================================================
# ADDRESS INDEXES
# =========================================================

s2_address_index_v2 = (
    s2.groupby("normalized_address_v2")["entity_id"]
    .agg(list)
    .to_dict()
)

s3_address_index_v2 = (
    s3.groupby("normalized_address_v2")["entity_id"]
    .agg(list)
    .to_dict()
)


print("S2 name index keys:",
      len(s2_name_index_v2))

print("S3 name index keys:",
      len(s3_name_index_v2))

print("S2 address index keys:",
      len(s2_address_index_v2))

print("S3 address index keys:",
      len(s3_address_index_v2))

Building V2 exact indexes...
S2 name index keys: 3946649
S3 name index keys: 4188983
S2 address index keys: 4082934
S3 address index keys: 4431038


In [ ]:
def generate_v2_exact_candidates(s1_row):
    """
    Candidate generation using only V2 exact normalized
    name and address matches.
    """

    candidates = set()

    name = s1_row["normalized_name_v2"]
    address = s1_row["normalized_address_v2"]

    if name:
        candidates.update(
            s2_name_index_v2.get(name, [])
        )
        candidates.update(
            s3_name_index_v2.get(name, [])
        )

    if address:
        candidates.update(
            s2_address_index_v2.get(address, [])
        )
        candidates.update(
            s3_address_index_v2.get(address, [])
        )

    return candidates

In [ ]:
# Add V2 normalized fields directly to validation S1

validation_eval_v2 = validation_eval.copy()

validation_eval_v2["normalized_name_v2"] = (
    validation_eval_v2["business_name"]
    .map(normalize_name_v2)
)

validation_eval_v2["normalized_address_v2"] = (
    validation_eval_v2["business_address"]
    .map(normalize_address_v2)
)

print(
    "Validation rows:",
    len(validation_eval_v2)
)

print(
    "Missing V2 names:",
    validation_eval_v2["normalized_name_v2"].eq("").sum()
)

print(
    "Missing V2 addresses:",
    validation_eval_v2["normalized_address_v2"].eq("").sum()
)

Validation rows: 441365
Missing V2 names: 0
Missing V2 addresses: 0


In [ ]:
# ---------------------------------------------------------
# Evaluate V2 exact blocking on full validation
# ---------------------------------------------------------

total_pairs = 0
recovered_pairs = 0

candidate_counts = []

complete_recall_count = 0
zero_candidate_count = 0

total_s2_pairs = 0
recovered_s2_pairs = 0

total_s3_pairs = 0
recovered_s3_pairs = 0


for _, row in validation_eval_v2[
    [
        "source1_entity_id",
        "normalized_name_v2",
        "normalized_address_v2",
        "matched_entity_ids",
    ]
].iterrows():

    gt_ids = set(
        parse_gt_ids(row["matched_entity_ids"])
    )

    candidates = generate_v2_exact_candidates(row)

    recovered = gt_ids & candidates

    total_pairs += len(gt_ids)
    recovered_pairs += len(recovered)

    candidate_counts.append(len(candidates))

    if len(gt_ids) == len(recovered):
        complete_recall_count += 1

    if len(candidates) == 0:
        zero_candidate_count += 1

    gt_s2 = {
        entity_id
        for entity_id in gt_ids
        if entity_id.startswith("S2-")
    }

    gt_s3 = {
        entity_id
        for entity_id in gt_ids
        if entity_id.startswith("S3-")
    }

    recovered_s2 = gt_s2 & candidates
    recovered_s3 = gt_s3 & candidates

    total_s2_pairs += len(gt_s2)
    recovered_s2_pairs += len(recovered_s2)

    total_s3_pairs += len(gt_s3)
    recovered_s3_pairs += len(recovered_s3)


validation_v2_exact_summary = pd.DataFrame([{
    "blocker": "V2 exact name/address",

    "total_s1": len(validation_eval_v2),

    "total_gt_pairs": total_pairs,

    "pair_recall": (
        recovered_pairs / total_pairs
        if total_pairs else 0
    ),

    "s2_pair_recall": (
        recovered_s2_pairs / total_s2_pairs
        if total_s2_pairs else 0
    ),

    "s3_pair_recall": (
        recovered_s3_pairs / total_s3_pairs
        if total_s3_pairs else 0
    ),

    "mean_candidates": np.mean(candidate_counts),

    "median_candidates": np.median(candidate_counts),

    "p90_candidates": np.percentile(
        candidate_counts, 90
    ),

    "p95_candidates": np.percentile(
        candidate_counts, 95
    ),

    "p99_candidates": np.percentile(
        candidate_counts, 99
    ),

    "max_candidates": np.max(candidate_counts),

    "zero_candidate_s1": zero_candidate_count,

    "zero_candidate_rate": (
        zero_candidate_count / len(validation_eval_v2)
    ),

    "complete_s1_recall": (
        complete_recall_count / len(validation_eval_v2)
    ),
}])

display(validation_v2_exact_summary)

,blocker,total_s1,total_gt_pairs,pair_recall,s2_pair_recall,s3_pair_recall,mean_candidates,median_candidates,p90_candidates,p95_candidates,p99_candidates,max_candidates,zero_candidate_s1,zero_candidate_rate,complete_s1_recall
0,V2 exact name/address,441365,1527679,0.350101,0.397665,0.305499,11.553893,2.0,18.0,70.0,192.0,508,82173,0.186179,0.120966


In [ ]:
# ---------------------------------------------------------
# Build V2 token frequencies
# ---------------------------------------------------------

from collections import Counter

print("Building V2 token frequencies...")


def get_unique_tokens(series):
    counter = Counter()

    for text in series:
        if not text:
            continue

        counter.update(
            set(text.split())
        )

    return counter


# =========================================================
# S2
# =========================================================

s2_name_token_counts_v2 = get_unique_tokens(
    s2["normalized_name_v2"]
)

s2_address_token_counts_v2 = get_unique_tokens(
    s2["normalized_address_v2"]
)


# =========================================================
# S3
# =========================================================

s3_name_token_counts_v2 = get_unique_tokens(
    s3["normalized_name_v2"]
)

s3_address_token_counts_v2 = get_unique_tokens(
    s3["normalized_address_v2"]
)


print()
print("S2 name tokens:", len(s2_name_token_counts_v2))
print("S2 address tokens:", len(s2_address_token_counts_v2))

print("S3 name tokens:", len(s3_name_token_counts_v2))
print("S3 address tokens:", len(s3_address_token_counts_v2))


# ---------------------------------------------------------
# Show the most frequent tokens
# ---------------------------------------------------------

print("\nTop S2 name tokens:")
print(s2_name_token_counts_v2.most_common(15))

print("\nTop S3 name tokens:")
print(s3_name_token_counts_v2.most_common(15))

print("\nTop S2 address tokens:")
print(s2_address_token_counts_v2.most_common(15))

print("\nTop S3 address tokens:")
print(s3_address_token_counts_v2.most_common(15))

Building V2 token frequencies...

S2 name tokens: 789500
S2 address tokens: 607847
S3 name tokens: 840942
S3 address tokens: 579983

Top S2 name tokens:
[('limited', 543250), ('private', 540952), ('llc', 528303), ('inc', 413268), ('ltd', 398464), ('ल', 230999), ('ट', 208088), ('ड', 203924), ('म', 203113), ('पर', 202694), ('com', 201501), ('center', 198410), ('pvt', 189270), ('इव', 157955), ('partners', 155561)]

Top S3 name tokens:
[('limited', 692890), ('private', 656710), ('llc', 571362), ('inc', 433684), ('ltd', 427781), ('center', 225939), ('pvt', 220353), ('com', 211159), ('services', 170272), ('partners', 165182), ('s', 147647), ('group', 147399), ('co', 146459), ('corp', 142555), ('ल', 130077)]

Top S2 address tokens:
[('road', 998770), ('no', 916456), ('street', 668885), ('drive', 499393), ('avenue', 417909), ('floor', 319586), ('maharashtra', 319026), ('city', 298415), ('tx', 292289), ('nagar', 286965), ('1', 281615), ('a', 266211), ('delhi', 262478), ('c', 254392), ('lane', 2

In [ ]:
# ---------------------------------------------------------
# Build V2 rare-token inverted indexes
# threshold = 500
# ---------------------------------------------------------

V2_TOKEN_THRESHOLD = 500

print(
    f"Building V2 token indexes with threshold "
    f"{V2_TOKEN_THRESHOLD}..."
)


def build_token_index(
    series,
    token_counts,
    threshold=500,
):
    index = defaultdict(list)

    for entity_id, text in zip(
        series.index,
        series,
    ):
        if not text:
            continue

        tokens = set(text.split())

        for token in tokens:
            if token_counts[token] <= threshold:
                index[token].append(
                    entity_id
                )

    return index

Building V2 token indexes with threshold 500...


In [ ]:
# ---------------------------------------------------------
# Build V2 rare-token inverted indexes
# threshold = 500
# ---------------------------------------------------------

V2_TOKEN_THRESHOLD = 500

print(
    f"Building V2 token indexes with threshold "
    f"{V2_TOKEN_THRESHOLD}..."
)


def build_token_index(
    entity_ids,
    normalized_text,
    token_counts,
    threshold=500,
):
    index = defaultdict(list)

    for entity_id, text in zip(
        entity_ids,
        normalized_text,
    ):
        if not text:
            continue

        tokens = set(text.split())

        for token in tokens:
            if token_counts[token] <= threshold:
                index[token].append(entity_id)

    return index


# ---------------------------------------------------------
# S2
# ---------------------------------------------------------

s2_name_token_index_v2 = build_token_index(
    s2["entity_id"],
    s2["normalized_name_v2"],
    s2_name_token_counts_v2,
    V2_TOKEN_THRESHOLD,
)

s2_address_token_index_v2 = build_token_index(
    s2["entity_id"],
    s2["normalized_address_v2"],
    s2_address_token_counts_v2,
    V2_TOKEN_THRESHOLD,
)


# ---------------------------------------------------------
# S3
# ---------------------------------------------------------

s3_name_token_index_v2 = build_token_index(
    s3["entity_id"],
    s3["normalized_name_v2"],
    s3_name_token_counts_v2,
    V2_TOKEN_THRESHOLD,
)

s3_address_token_index_v2 = build_token_index(
    s3["entity_id"],
    s3["normalized_address_v2"],
    s3_address_token_counts_v2,
    V2_TOKEN_THRESHOLD,
)


print()
print(
    "S2 name index keys:",
    len(s2_name_token_index_v2)
)

print(
    "S2 address index keys:",
    len(s2_address_token_index_v2)
)

print(
    "S3 name index keys:",
    len(s3_name_token_index_v2)
)

print(
    "S3 address index keys:",
    len(s3_address_token_index_v2)
)

Building V2 token indexes with threshold 500...

S2 name index keys: 786551
S2 address index keys: 601782
S3 name index keys: 838164
S3 address index keys: 573940


In [ ]:
V2_TOKEN_THRESHOLD = 500
V2_TOP_K = 2


def generate_v2_block_candidates(s1_row, threshold=500, top_k=2):
    """
    Generate S2/S3 candidates using V2 normalization.

    Blocking rules:
      1. Exact normalized V2 name
      2. Exact normalized V2 address
      3. Top-k rarest eligible V2 name tokens
      4. Top-k rarest eligible V2 address tokens
    """

    candidates = set()

    # --------------------------------------------------
    # 1. Exact V2 name
    # --------------------------------------------------
    normalized_name = s1_row["normalized_name_v2"]

    if normalized_name:
        candidates.update(
            s2_name_index_v2.get(normalized_name, [])
        )
        candidates.update(
            s3_name_index_v2.get(normalized_name, [])
        )

    # --------------------------------------------------
    # 2. Exact V2 address
    # --------------------------------------------------
    normalized_address = s1_row["normalized_address_v2"]

    if normalized_address:
        candidates.update(
            s2_address_index_v2.get(normalized_address, [])
        )
        candidates.update(
            s3_address_index_v2.get(normalized_address, [])
        )

    # --------------------------------------------------
    # 3. Rarest V2 NAME tokens
    # --------------------------------------------------
    if normalized_name:

        name_tokens = set(normalized_name.split())

        # S2
        s2_name_tokens = [
            (token, s2_name_token_counts_v2[token])
            for token in name_tokens
            if (
                token in s2_name_token_counts_v2
                and s2_name_token_counts_v2[token] <= threshold
            )
        ]

        s2_name_tokens.sort(key=lambda x: x[1])

        for token, _ in s2_name_tokens[:top_k]:
            candidates.update(
                s2_name_token_index_v2.get(token, [])
            )

        # S3
        s3_name_tokens = [
            (token, s3_name_token_counts_v2[token])
            for token in name_tokens
            if (
                token in s3_name_token_counts_v2
                and s3_name_token_counts_v2[token] <= threshold
            )
        ]

        s3_name_tokens.sort(key=lambda x: x[1])

        for token, _ in s3_name_tokens[:top_k]:
            candidates.update(
                s3_name_token_index_v2.get(token, [])
            )

    # --------------------------------------------------
    # 4. Rarest V2 ADDRESS tokens
    # --------------------------------------------------
    if normalized_address:

        address_tokens = set(normalized_address.split())

        # S2
        s2_address_tokens = [
            (token, s2_address_token_counts_v2[token])
            for token in address_tokens
            if (
                token in s2_address_token_counts_v2
                and s2_address_token_counts_v2[token] <= threshold
            )
        ]

        s2_address_tokens.sort(key=lambda x: x[1])

        for token, _ in s2_address_tokens[:top_k]:
            candidates.update(
                s2_address_token_index_v2.get(token, [])
            )

        # S3
        s3_address_tokens = [
            (token, s3_address_token_counts_v2[token])
            for token in address_tokens
            if (
                token in s3_address_token_counts_v2
                and s3_address_token_counts_v2[token] <= threshold
            )
        ]

        s3_address_tokens.sort(key=lambda x: x[1])

        for token, _ in s3_address_tokens[:top_k]:
            candidates.update(
                s3_address_token_index_v2.get(token, [])
            )

    return candidates

In [ ]:
print(validation_s1.columns.tolist())
print(s1.columns.tolist())

['source1_entity_id', 'business_name', 'business_address', 'country', 'normalized_name', 'normalized_address']
['entity_id', 'business_name', 'business_address', 'country', 'normalized_name_v2', 'normalized_address_v2']


In [ ]:
# Add V2 normalized fields to validation_s1
v2_s1_features = s1[
    [
        "entity_id",
        "normalized_name_v2",
        "normalized_address_v2",
    ]
].rename(
    columns={
        "entity_id": "source1_entity_id"
    }
)

validation_s1 = validation_s1.merge(
    v2_s1_features,
    on="source1_entity_id",
    how="left",
    validate="one_to_one",
)

print("Validation shape:", validation_s1.shape)

print(
    validation_s1[
        ["normalized_name_v2", "normalized_address_v2"]
    ].isna().sum()
)

Validation shape: (441365, 8)
normalized_name_v2       0
normalized_address_v2    0
dtype: int64


In [ ]:
test_row = validation_s1.iloc[0]

test_candidates = generate_v2_block_candidates(
    test_row,
    threshold=V2_TOKEN_THRESHOLD,
    top_k=V2_TOP_K,
)

print("S1:", test_row["source1_entity_id"])
print("V2 candidate count:", len(test_candidates))
print("First 20 candidates:", list(test_candidates)[:20])

S1: S1-392794459
V2 candidate count: 2298
First 20 candidates: ['S2-974357108', 'S3-648827143', 'S2-691641626', 'S2-573897508', 'S2-5778408', 'S3-657460581', 'S3-137458497', 'S3-540543859', 'S3-635839799', 'S2-239583320', 'S2-309817989', 'S3-92759088', 'S3-542169159', 'S2-338453075', 'S2-461219137', 'S3-999704781', 'S3-34980491', 'S2-277294125', 'S2-424190137', 'S2-293240316']


In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
import time


# ------------------------------------------------------------
# Build validation ground-truth lookup
# ------------------------------------------------------------

valid_gt = (
    train_gt[
        train_gt["source1_entity_id"].isin(
            validation_s1["source1_entity_id"]
        )
    ]
    .copy()
)

valid_gt_lookup = dict(
    zip(
        valid_gt["source1_entity_id"],
        valid_gt["matched_entity_ids"],
    )
)


def parse_gt_ids(value):
    if pd.isna(value):
        return set()

    value = str(value).strip()

    if not value:
        return set()

    return {
        x.strip()
        for x in value.split(",")
        if x.strip()
    }


# ------------------------------------------------------------
# Evaluate blocker
# ------------------------------------------------------------

start_time = time.time()

total_gt_pairs = 0
recovered_pairs = 0

s2_gt_pairs = 0
s2_recovered_pairs = 0

s3_gt_pairs = 0
s3_recovered_pairs = 0

complete_s1_matches = 0
zero_candidate_s1 = 0

candidate_counts = []

validation_candidate_records = []


for row in validation_s1.itertuples(index=False):

    s1_id = row.source1_entity_id

    candidates = generate_v2_block_candidates(
        row._asdict(),
        threshold=V2_TOKEN_THRESHOLD,
        top_k=V2_TOP_K,
    )

    candidate_count = len(candidates)
    candidate_counts.append(candidate_count)

    if candidate_count == 0:
        zero_candidate_s1 += 1

    gt_ids = parse_gt_ids(
        valid_gt_lookup[s1_id]
    )

    total_gt_pairs += len(gt_ids)

    recovered = gt_ids.intersection(candidates)

    recovered_pairs += len(recovered)

    # Source-specific recall
    gt_s2 = {
        x for x in gt_ids
        if x.startswith("S2-")
    }

    gt_s3 = {
        x for x in gt_ids
        if x.startswith("S3-")
    }

    recovered_s2 = recovered.intersection(gt_s2)
    recovered_s3 = recovered.intersection(gt_s3)

    s2_gt_pairs += len(gt_s2)
    s2_recovered_pairs += len(recovered_s2)

    s3_gt_pairs += len(gt_s3)
    s3_recovered_pairs += len(recovered_s3)

    if gt_ids.issubset(candidates):
        complete_s1_matches += 1

    validation_candidate_records.append(
        {
            "source1_entity_id": s1_id,
            "candidate_count": candidate_count,
            "ground_truth_count": len(gt_ids),
            "recovered_count": len(recovered),
        }
    )


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

candidate_counts = np.array(candidate_counts)

pair_recall = (
    recovered_pairs / total_gt_pairs
    if total_gt_pairs
    else 0
)

s2_recall = (
    s2_recovered_pairs / s2_gt_pairs
    if s2_gt_pairs
    else 0
)

s3_recall = (
    s3_recovered_pairs / s3_gt_pairs
    if s3_gt_pairs
    else 0
)

complete_s1_recall = (
    complete_s1_matches / len(validation_s1)
)

zero_candidate_rate = (
    zero_candidate_s1 / len(validation_s1)
)


print("=" * 60)
print("V2 BLOCKER — FULL VALIDATION")
print("=" * 60)

print(f"Validation S1 rows:       {len(validation_s1):,}")
print(f"Total GT pairs:           {total_gt_pairs:,}")
print(f"Recovered GT pairs:       {recovered_pairs:,}")
print()

print(f"Overall pair recall:      {pair_recall:.6f}")
print(f"S2 pair recall:           {s2_recall:.6f}")
print(f"S3 pair recall:           {s3_recall:.6f}")
print()

print(f"Complete S1 recall:       {complete_s1_recall:.6f}")
print(f"Zero-candidate S1:        {zero_candidate_s1:,}")
print(f"Zero-candidate rate:      {zero_candidate_rate:.6f}")
print()

print(f"Mean candidates:          {candidate_counts.mean():.2f}")
print(f"Median candidates:        {np.median(candidate_counts):.0f}")
print(f"P90 candidates:           {np.percentile(candidate_counts, 90):.0f}")
print(f"P95 candidates:           {np.percentile(candidate_counts, 95):.0f}")
print(f"P99 candidates:           {np.percentile(candidate_counts, 99):.0f}")
print(f"Maximum candidates:       {candidate_counts.max():,}")
print()

print(f"Elapsed time:             {time.time() - start_time:.2f} sec")
print("=" * 60)

KeyError: 'S1-392794459'

In [ ]:
print("validation_s1 rows:", len(validation_s1))
print("train_gt rows:", len(train_gt))

print("\nValidation ID example:")
print(validation_s1["source1_entity_id"].iloc[0])

print("\ntrain_gt columns:")
print(train_gt.columns.tolist())

print("\ntrain_gt ID example:")
print(train_gt["source1_entity_id"].iloc[0])

print("\nDoes validation ID exist in train_gt?")
test_id = validation_s1["source1_entity_id"].iloc[0]
print(
    test_id in set(train_gt["source1_entity_id"])
)

print("\nMatching rows:")
print(
    train_gt[
        train_gt["source1_entity_id"] == test_id
    ]
)

validation_s1 rows: 441365
train_gt rows: 1765456

Validation ID example:
S1-392794459

train_gt columns:
['source1_entity_id', 'matched_entity_ids', 'match_count']

train_gt ID example:
S1-257620181

Does validation ID exist in train_gt?
False

Matching rows:
Empty DataFrame
Columns: [source1_entity_id, matched_entity_ids, match_count]
Index: []


In [ ]:
# Load the ORIGINAL full ground truth
full_gt = pd.read_csv(
    GT_PATH,
    sep="\t",
    dtype=str,
)

print("Full GT rows:", len(full_gt))
print("Columns:", full_gt.columns.tolist())

Full GT rows: 2206821
Columns: ['source1_entity_id', 'matched_entity_ids']


In [ ]:
# Build validation ground-truth lookup from FULL GT
valid_gt = full_gt[
    full_gt["source1_entity_id"].isin(
        validation_s1["source1_entity_id"]
    )
].copy()

valid_gt_lookup = dict(
    zip(
        valid_gt["source1_entity_id"],
        valid_gt["matched_entity_ids"],
    )
)

print("Validation GT rows:", len(valid_gt))
print("Validation S1 rows:", len(validation_s1))

test_id = validation_s1["source1_entity_id"].iloc[0]

print("\nTest S1:", test_id)
print("GT exists:", test_id in valid_gt_lookup)
print("GT:", valid_gt_lookup[test_id])

Validation GT rows: 441365
Validation S1 rows: 441365

Test S1: S1-392794459
GT exists: True
GT: S2-520412140,S2-614475791,S2-23296720,S3-696455298


In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
import time


# ------------------------------------------------------------
# Ground-truth lookup
# ------------------------------------------------------------

def parse_gt_ids(value):
    if pd.isna(value):
        return set()

    value = str(value).strip()

    if not value:
        return set()

    return {
        x.strip()
        for x in value.split(",")
        if x.strip()
    }


# ------------------------------------------------------------
# Full validation evaluation
# ------------------------------------------------------------

start_time = time.time()

total_gt_pairs = 0
recovered_pairs = 0

s2_gt_pairs = 0
s2_recovered_pairs = 0

s3_gt_pairs = 0
s3_recovered_pairs = 0

complete_s1_matches = 0
zero_candidate_s1 = 0

candidate_counts = []

validation_candidate_records = []


for row in validation_s1.itertuples(index=False):

    s1_id = row.source1_entity_id

    # itertuples() -> dict so the blocker can use column names
    row_dict = row._asdict()

    candidates = generate_v2_block_candidates(
        row_dict,
        threshold=V2_TOKEN_THRESHOLD,
        top_k=V2_TOP_K,
    )

    candidate_count = len(candidates)
    candidate_counts.append(candidate_count)

    if candidate_count == 0:
        zero_candidate_s1 += 1

    # Ground truth
    gt_ids = parse_gt_ids(
        valid_gt_lookup[s1_id]
    )

    total_gt_pairs += len(gt_ids)

    recovered = gt_ids.intersection(candidates)

    recovered_pairs += len(recovered)

    # Source-specific
    gt_s2 = {
        x for x in gt_ids
        if x.startswith("S2-")
    }

    gt_s3 = {
        x for x in gt_ids
        if x.startswith("S3-")
    }

    recovered_s2 = recovered.intersection(gt_s2)
    recovered_s3 = recovered.intersection(gt_s3)

    s2_gt_pairs += len(gt_s2)
    s2_recovered_pairs += len(recovered_s2)

    s3_gt_pairs += len(gt_s3)
    s3_recovered_pairs += len(recovered_s3)

    # Did we recover EVERY true target for this S1?
    if gt_ids.issubset(candidates):
        complete_s1_matches += 1

    validation_candidate_records.append(
        {
            "source1_entity_id": s1_id,
            "candidate_count": candidate_count,
            "ground_truth_count": len(gt_ids),
            "recovered_count": len(recovered),
        }
    )


# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------

candidate_counts = np.asarray(candidate_counts)

pair_recall = (
    recovered_pairs / total_gt_pairs
)

s2_recall = (
    s2_recovered_pairs / s2_gt_pairs
)

s3_recall = (
    s3_recovered_pairs / s3_gt_pairs
)

complete_s1_recall = (
    complete_s1_matches / len(validation_s1)
)

zero_candidate_rate = (
    zero_candidate_s1 / len(validation_s1)
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("=" * 65)
print("V2 BLOCKER — FULL VALIDATION")
print("=" * 65)

print(f"Validation S1 rows:       {len(validation_s1):,}")
print(f"Total GT pairs:           {total_gt_pairs:,}")
print(f"Recovered GT pairs:       {recovered_pairs:,}")
print()

print(f"Overall pair recall:      {pair_recall:.6f}")
print(f"S2 pair recall:           {s2_recall:.6f}")
print(f"S3 pair recall:           {s3_recall:.6f}")
print()

print(f"Complete S1 recall:       {complete_s1_recall:.6f}")
print(f"Zero-candidate S1:        {zero_candidate_s1:,}")
print(f"Zero-candidate rate:      {zero_candidate_rate:.6f}")
print()

print(f"Mean candidates:          {candidate_counts.mean():.2f}")
print(f"Median candidates:        {np.median(candidate_counts):.0f}")
print(f"P90 candidates:           {np.percentile(candidate_counts, 90):.0f}")
print(f"P95 candidates:            {np.percentile(candidate_counts, 95):.0f}")
print(f"P99 candidates:           {np.percentile(candidate_counts, 99):.0f}")
print(f"Maximum candidates:       {candidate_counts.max():,}")
print()

print(f"Elapsed time:             {time.time() - start_time:.2f} sec")
print("=" * 65)

V2 BLOCKER — FULL VALIDATION
Validation S1 rows:       441,365
Total GT pairs:           1,527,679
Recovered GT pairs:       1,315,597

Overall pair recall:      0.861174
S2 pair recall:           0.871966
S3 pair recall:           0.851054

Complete S1 recall:       0.735847
Zero-candidate S1:        6,120
Zero-candidate rate:      0.013866

Mean candidates:          464.28
Median candidates:        374
P90 candidates:           1037
P95 candidates:            1260
P99 candidates:           1688
Maximum candidates:       3,112

Elapsed time:             62.92 sec


In [ ]:
# ============================================================
# STEP 1 — BUILD MISSED TRUE-PAIR FORENSIC TABLE
# ============================================================

# Convert validation candidate records to DataFrame
validation_candidate_df = pd.DataFrame(
    validation_candidate_records
)

print("Candidate evaluation rows:", len(validation_candidate_df))

Candidate evaluation rows: 441365


In [ ]:
# ============================================================
# Re-run validation blocker and store candidate IDs
# ============================================================

start_time = time.time()

missed_pairs = []

for row in validation_s1.itertuples(index=False):

    s1_id = row.source1_entity_id

    candidates = generate_v2_block_candidates(
        row._asdict(),
        threshold=V2_TOKEN_THRESHOLD,
        top_k=V2_TOP_K,
    )

    gt_ids = parse_gt_ids(
        valid_gt_lookup[s1_id]
    )

    missed = gt_ids - candidates

    for target_id in missed:
        missed_pairs.append(
            {
                "source1_entity_id": s1_id,
                "target_entity_id": target_id,
            }
        )

print(f"Missed pairs: {len(missed_pairs):,}")
print(f"Elapsed: {time.time() - start_time:.2f} sec")

Missed pairs: 212,082
Elapsed: 110.08 sec


In [ ]:
# ============================================================
# Attach S2/S3 target records
# ============================================================

missed_df = pd.DataFrame(missed_pairs)

missed_s2 = (
    missed_df[
        missed_df["target_entity_id"].str.startswith("S2-")
    ]
    .merge(
        s2[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country",
                "normalized_name_v2",
                "normalized_address_v2",
            ]
        ],
        left_on="target_entity_id",
        right_on="entity_id",
        how="left",
    )
    .drop(columns=["entity_id"])
)

missed_s3 = (
    missed_df[
        missed_df["target_entity_id"].str.startswith("S3-")
    ]
    .merge(
        s3[
            [
                "entity_id",
                "business_name",
                "business_address",
                "country",
                "normalized_name_v2",
                "normalized_address_v2",
            ]
        ],
        left_on="target_entity_id",
        right_on="entity_id",
        how="left",
    )
    .drop(columns=["entity_id"])
)

missed_targets = pd.concat(
    [missed_s2, missed_s3],
    ignore_index=True,
)

print("Missed S2 pairs:", len(missed_s2))
print("Missed S3 pairs:", len(missed_s3))
print("Total:", len(missed_targets))

Missed S2 pairs: 94655
Missed S3 pairs: 117427
Total: 212082


In [ ]:
# ============================================================
# Attach S1 records
# ============================================================

missed_targets = missed_targets.merge(
    validation_s1[
        [
            "source1_entity_id",
            "business_name",
            "business_address",
            "country",
            "normalized_name_v2",
            "normalized_address_v2",
        ]
    ].rename(
        columns={
            "business_name": "s1_business_name",
            "business_address": "s1_business_address",
            "country": "s1_country",
            "normalized_name_v2": "s1_normalized_name_v2",
            "normalized_address_v2": "s1_normalized_address_v2",
        }
    ),
    on="source1_entity_id",
    how="left",
)

missed_targets = missed_targets.rename(
    columns={
        "business_name": "target_business_name",
        "business_address": "target_business_address",
        "country": "target_country",
        "normalized_name_v2": "target_normalized_name_v2",
        "normalized_address_v2": "target_normalized_address_v2",
    }
)

print(missed_targets.shape)

(212082, 12)


In [ ]:
# ============================================================
# STEP 4 — TOKEN OVERLAP DIAGNOSTICS
# ============================================================

def token_set(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return set(str(value).split())


def token_overlap(row):
    s1_name = token_set(row["s1_normalized_name_v2"])
    tgt_name = token_set(row["target_normalized_name_v2"])

    s1_addr = token_set(row["s1_normalized_address_v2"])
    tgt_addr = token_set(row["target_normalized_address_v2"])

    return pd.Series(
        {
            "name_shared_tokens": len(s1_name & tgt_name),
            "address_shared_tokens": len(s1_addr & tgt_addr),
            "name_token_jaccard": (
                len(s1_name & tgt_name) /
                len(s1_name | tgt_name)
                if (s1_name | tgt_name)
                else 0
            ),
            "address_token_jaccard": (
                len(s1_addr & tgt_addr) /
                len(s1_addr | tgt_addr)
                if (s1_addr | tgt_addr)
                else 0
            ),
        }
    )


overlap_features = missed_targets.apply(
    token_overlap,
    axis=1,
)

missed_targets = pd.concat(
    [missed_targets, overlap_features],
    axis=1,
)

print(
    missed_targets[
        [
            "name_shared_tokens",
            "address_shared_tokens",
            "name_token_jaccard",
            "address_token_jaccard",
        ]
    ].describe()
)

       name_shared_tokens  address_shared_tokens  name_token_jaccard  \
count       212082.000000          212082.000000       212082.000000   
mean             1.810460               4.727785            0.429159   
std              1.454024               3.045472            0.332181   
min              0.000000               0.000000            0.000000   
25%              0.000000               3.000000            0.000000   
50%              2.000000               4.000000            0.500000   
75%              3.000000               6.000000            0.666667   
max              9.000000              24.000000            1.000000   

       address_token_jaccard  
count          212082.000000  
mean                0.491041  
std                 0.272959  
min                 0.000000  
25%                 0.300000  
50%                 0.500000  
75%                 0.700000  
max                 1.000000  


In [ ]:
# ============================================================
# WHY DID V2 MISS THE PAIRS?
# ============================================================

name_zero = (
    missed_targets["name_shared_tokens"] == 0
)

address_zero = (
    missed_targets["address_shared_tokens"] == 0
)

print("Total missed pairs:", len(missed_targets))

print()
print(
    "No shared NAME token:",
    name_zero.sum(),
    f"({name_zero.mean():.2%})"
)

print(
    "No shared ADDRESS token:",
    address_zero.sum(),
    f"({address_zero.mean():.2%})"
)

print(
    "No shared NAME OR ADDRESS token:",
    (name_zero & address_zero).sum(),
    f"({(name_zero & address_zero).mean():.2%})"
)

print(
    "Shared name token:",
    (~name_zero).sum(),
    f"({(~name_zero).mean():.2%})"
)

print(
    "Shared address token:",
    (~address_zero).sum(),
    f"({(~address_zero).mean():.2%})"
)

Total missed pairs: 212082

No shared NAME token: 67739 (31.94%)
No shared ADDRESS token: 23246 (10.96%)
No shared NAME OR ADDRESS token: 5 (0.00%)
Shared name token: 144343 (68.06%)
Shared address token: 188836 (89.04%)


In [ ]:
# ============================================================
# V2 BLOCKER — K=3
# ============================================================

V2_K3 = 3

start_time = time.time()

total_gt_pairs = 0
recovered_pairs = 0

s2_gt_pairs = 0
s2_recovered_pairs = 0

s3_gt_pairs = 0
s3_recovered_pairs = 0

complete_s1_matches = 0
zero_candidate_s1 = 0

candidate_counts_k3 = []


for row in validation_s1.itertuples(index=False):

    s1_id = row.source1_entity_id

    candidates = generate_v2_block_candidates(
        row._asdict(),
        threshold=V2_TOKEN_THRESHOLD,
        top_k=V2_K3,
    )

    candidate_count = len(candidates)
    candidate_counts_k3.append(candidate_count)

    if candidate_count == 0:
        zero_candidate_s1 += 1

    gt_ids = parse_gt_ids(
        valid_gt_lookup[s1_id]
    )

    total_gt_pairs += len(gt_ids)

    recovered = gt_ids.intersection(candidates)

    recovered_pairs += len(recovered)

    gt_s2 = {
        x for x in gt_ids
        if x.startswith("S2-")
    }

    gt_s3 = {
        x for x in gt_ids
        if x.startswith("S3-")
    }

    recovered_s2 = recovered.intersection(gt_s2)
    recovered_s3 = recovered.intersection(gt_s3)

    s2_gt_pairs += len(gt_s2)
    s2_recovered_pairs += len(recovered_s2)

    s3_gt_pairs += len(gt_s3)
    s3_recovered_pairs += len(recovered_s3)

    if gt_ids.issubset(candidates):
        complete_s1_matches += 1


candidate_counts_k3 = np.asarray(candidate_counts_k3)

pair_recall_k3 = recovered_pairs / total_gt_pairs
s2_recall_k3 = s2_recovered_pairs / s2_gt_pairs
s3_recall_k3 = s3_recovered_pairs / s3_gt_pairs

complete_s1_recall_k3 = (
    complete_s1_matches / len(validation_s1)
)

zero_candidate_rate_k3 = (
    zero_candidate_s1 / len(validation_s1)
)


print("=" * 65)
print("V2 BLOCKER — K=3 — FULL VALIDATION")
print("=" * 65)

print(f"Validation S1 rows:       {len(validation_s1):,}")
print(f"Total GT pairs:           {total_gt_pairs:,}")
print(f"Recovered GT pairs:       {recovered_pairs:,}")
print()

print(f"Overall pair recall:      {pair_recall_k3:.6f}")
print(f"S2 pair recall:           {s2_recall_k3:.6f}")
print(f"S3 pair recall:           {s3_recall_k3:.6f}")
print()

print(f"Complete S1 recall:       {complete_s1_recall_k3:.6f}")
print(f"Zero-candidate S1:        {zero_candidate_s1:,}")
print(f"Zero-candidate rate:      {zero_candidate_rate_k3:.6f}")
print()

print(f"Mean candidates:          {candidate_counts_k3.mean():.2f}")
print(f"Median candidates:        {np.median(candidate_counts_k3):.0f}")
print(f"P90 candidates:           {np.percentile(candidate_counts_k3, 90):.0f}")
print(f"P95 candidates:           {np.percentile(candidate_counts_k3, 95):.0f}")
print(f"P99 candidates:           {np.percentile(candidate_counts_k3, 99):.0f}")
print(f"Maximum candidates:       {candidate_counts_k3.max():,}")
print()

print(f"Elapsed time:             {time.time() - start_time:.2f} sec")
print("=" * 65)

V2 BLOCKER — K=3 — FULL VALIDATION
Validation S1 rows:       441,365
Total GT pairs:           1,527,679
Recovered GT pairs:       1,318,658

Overall pair recall:      0.863177
S2 pair recall:           0.873677
S3 pair recall:           0.853332

Complete S1 recall:       0.740319
Zero-candidate S1:        6,120
Zero-candidate rate:      0.013866

Mean candidates:          523.15
Median candidates:        438
P90 candidates:           1154
P95 candidates:           1399
P99 candidates:           1873
Maximum candidates:       3,768

Elapsed time:             137.15 sec


In [ ]:
def generate_v2_block_candidates_split_k(
    s1_row,
    threshold=500,
    name_top_k=2,
    address_top_k=3,
):
    """
    V2 blocker with separate K values for name and address.

    Blocks:
      1. Exact normalized name
      2. Exact normalized address
      3. Rarest name tokens
      4. Rarest address tokens
    """

    candidates = set()

    normalized_name = s1_row["normalized_name_v2"]
    normalized_address = s1_row["normalized_address_v2"]

    # ========================================================
    # 1. Exact normalized name
    # ========================================================

    if normalized_name:
        candidates.update(
            s2_name_index_v2.get(normalized_name, [])
        )
        candidates.update(
            s3_name_index_v2.get(normalized_name, [])
        )

    # ========================================================
    # 2. Exact normalized address
    # ========================================================

    if normalized_address:
        candidates.update(
            s2_address_index_v2.get(normalized_address, [])
        )
        candidates.update(
            s3_address_index_v2.get(normalized_address, [])
        )

    # ========================================================
    # 3. Rare name tokens
    # ========================================================

    if normalized_name:

        name_tokens = set(normalized_name.split())

        # ---------- S2 ----------

        s2_name_tokens = [
            (token, s2_name_token_counts_v2[token])
            for token in name_tokens
            if (
                token in s2_name_token_counts_v2
                and s2_name_token_counts_v2[token] <= threshold
            )
        ]

        s2_name_tokens.sort(key=lambda x: x[1])

        for token, _ in s2_name_tokens[:name_top_k]:
            candidates.update(
                s2_name_token_index_v2.get(token, [])
            )

        # ---------- S3 ----------

        s3_name_tokens = [
            (token, s3_name_token_counts_v2[token])
            for token in name_tokens
            if (
                token in s3_name_token_counts_v2
                and s3_name_token_counts_v2[token] <= threshold
            )
        ]

        s3_name_tokens.sort(key=lambda x: x[1])

        for token, _ in s3_name_tokens[:name_top_k]:
            candidates.update(
                s3_name_token_index_v2.get(token, [])
            )

    # ========================================================
    # 4. Rare address tokens
    # ========================================================

    if normalized_address:

        address_tokens = set(normalized_address.split())

        # ---------- S2 ----------

        s2_address_tokens = [
            (token, s2_address_token_counts_v2[token])
            for token in address_tokens
            if (
                token in s2_address_token_counts_v2
                and s2_address_token_counts_v2[token] <= threshold
            )
        ]

        s2_address_tokens.sort(key=lambda x: x[1])

        for token, _ in s2_address_tokens[:address_top_k]:
            candidates.update(
                s2_address_token_index_v2.get(token, [])
            )

        # ---------- S3 ----------

        s3_address_tokens = [
            (token, s3_address_token_counts_v2[token])
            for token in address_tokens
            if (
                token in s3_address_token_counts_v2
                and s3_address_token_counts_v2[token] <= threshold
            )
        ]

        s3_address_tokens.sort(key=lambda x: x[1])

        for token, _ in s3_address_tokens[:address_top_k]:
            candidates.update(
                s3_address_token_index_v2.get(token, [])
            )

    return candidates

In [ ]:
test_row = validation_s1.iloc[0]

test_candidates = generate_v2_block_candidates_split_k(
    test_row,
    threshold=500,
    name_top_k=2,
    address_top_k=3,
)

print("S1:", test_row["source1_entity_id"])
print("Candidate count:", len(test_candidates))
print("First 20:", list(test_candidates)[:20])

S1: S1-392794459
Candidate count: 2298
First 20: ['S2-974357108', 'S3-648827143', 'S2-691641626', 'S2-573897508', 'S2-5778408', 'S3-657460581', 'S3-137458497', 'S3-540543859', 'S3-635839799', 'S2-239583320', 'S2-309817989', 'S3-92759088', 'S3-542169159', 'S2-338453075', 'S2-461219137', 'S3-999704781', 'S3-34980491', 'S2-277294125', 'S2-424190137', 'S2-293240316']


In [ ]:
# ============================================================
# V2 BLOCKER — NAME K=2, ADDRESS K=3
# ============================================================

NAME_K = 2
ADDRESS_K = 3

start_time = time.time()

total_gt_pairs = 0
recovered_pairs = 0

s2_gt_pairs = 0
s2_recovered_pairs = 0

s3_gt_pairs = 0
s3_recovered_pairs = 0

complete_s1_matches = 0
zero_candidate_s1 = 0

candidate_counts_split_k = []


for row in validation_s1.itertuples(index=False):

    s1_id = row.source1_entity_id

    candidates = generate_v2_block_candidates_split_k(
        row._asdict(),
        threshold=V2_TOKEN_THRESHOLD,
        name_top_k=NAME_K,
        address_top_k=ADDRESS_K,
    )

    candidate_count = len(candidates)
    candidate_counts_split_k.append(candidate_count)

    if candidate_count == 0:
        zero_candidate_s1 += 1

    gt_ids = parse_gt_ids(
        valid_gt_lookup[s1_id]
    )

    total_gt_pairs += len(gt_ids)

    recovered = gt_ids.intersection(candidates)

    recovered_pairs += len(recovered)

    # --------------------------------------------------------
    # S2 / S3 breakdown
    # --------------------------------------------------------

    gt_s2 = {
        x for x in gt_ids
        if x.startswith("S2-")
    }

    gt_s3 = {
        x for x in gt_ids
        if x.startswith("S3-")
    }

    recovered_s2 = recovered.intersection(gt_s2)
    recovered_s3 = recovered.intersection(gt_s3)

    s2_gt_pairs += len(gt_s2)
    s2_recovered_pairs += len(recovered_s2)

    s3_gt_pairs += len(gt_s3)
    s3_recovered_pairs += len(recovered_s3)

    # --------------------------------------------------------
    # Complete S1 recall
    # --------------------------------------------------------

    if gt_ids.issubset(candidates):
        complete_s1_matches += 1


candidate_counts_split_k = np.asarray(
    candidate_counts_split_k
)

pair_recall_split_k = (
    recovered_pairs / total_gt_pairs
)

s2_recall_split_k = (
    s2_recovered_pairs / s2_gt_pairs
)

s3_recall_split_k = (
    s3_recovered_pairs / s3_gt_pairs
)

complete_s1_recall_split_k = (
    complete_s1_matches / len(validation_s1)
)

zero_candidate_rate_split_k = (
    zero_candidate_s1 / len(validation_s1)
)


print("=" * 70)
print("V2 BLOCKER — NAME K=2 / ADDRESS K=3")
print("=" * 70)

print(f"Validation S1 rows:       {len(validation_s1):,}")
print(f"Total GT pairs:           {total_gt_pairs:,}")
print(f"Recovered GT pairs:       {recovered_pairs:,}")
print()

print(f"Overall pair recall:      {pair_recall_split_k:.6f}")
print(f"S2 pair recall:           {s2_recall_split_k:.6f}")
print(f"S3 pair recall:           {s3_recall_split_k:.6f}")
print()

print(
    f"Complete S1 recall:       "
    f"{complete_s1_recall_split_k:.6f}"
)

print(f"Zero-candidate S1:        {zero_candidate_s1:,}")
print(
    f"Zero-candidate rate:      "
    f"{zero_candidate_rate_split_k:.6f}"
)

print()

print(
    f"Mean candidates:          "
    f"{candidate_counts_split_k.mean():.2f}"
)

print(
    f"Median candidates:        "
    f"{np.median(candidate_counts_split_k):.0f}"
)

print(
    f"P90 candidates:           "
    f"{np.percentile(candidate_counts_split_k, 90):.0f}"
)

print(
    f"P95 candidates:           "
    f"{np.percentile(candidate_counts_split_k, 95):.0f}"
)

print(
    f"P99 candidates:           "
    f"{np.percentile(candidate_counts_split_k, 99):.0f}"
)

print(
    f"Maximum candidates:       "
    f"{candidate_counts_split_k.max():,}"
)

print()

print(
    f"Elapsed time:             "
    f"{time.time() - start_time:.2f} sec"
)

print("=" * 70)

V2 BLOCKER — NAME K=2 / ADDRESS K=3
Validation S1 rows:       441,365
Total GT pairs:           1,527,679
Recovered GT pairs:       1,318,650

Overall pair recall:      0.863172
S2 pair recall:           0.873671
S3 pair recall:           0.853327

Complete S1 recall:       0.740303
Zero-candidate S1:        6,120
Zero-candidate rate:      0.013866

Mean candidates:          519.72
Median candidates:        436
P90 candidates:           1146
P95 candidates:           1388
P99 candidates:           1850
Maximum candidates:       3,357

Elapsed time:             74.90 sec


In [ ]:
# ============================================================
# V2 TOKEN INDEX — THRESHOLD 1000
# ============================================================

V2_TOKEN_THRESHOLD_1000 = 1000

print("Building V2 token indexes with threshold 1000...")

s2_name_token_index_v2_1000 = build_token_index(
    s2["entity_id"],
    s2["normalized_name_v2"],
    s2_name_token_counts_v2,
    V2_TOKEN_THRESHOLD_1000,
)

s2_address_token_index_v2_1000 = build_token_index(
    s2["entity_id"],
    s2["normalized_address_v2"],
    s2_address_token_counts_v2,
    V2_TOKEN_THRESHOLD_1000,
)

s3_name_token_index_v2_1000 = build_token_index(
    s3["entity_id"],
    s3["normalized_name_v2"],
    s3_name_token_counts_v2,
    V2_TOKEN_THRESHOLD_1000,
)

s3_address_token_index_v2_1000 = build_token_index(
    s3["entity_id"],
    s3["normalized_address_v2"],
    s3_address_token_counts_v2,
    V2_TOKEN_THRESHOLD_1000,
)

print()
print("S2 name index keys:",
      len(s2_name_token_index_v2_1000))

print("S2 address index keys:",
      len(s2_address_token_index_v2_1000))

print("S3 name index keys:",
      len(s3_name_token_index_v2_1000))

print("S3 address index keys:",
      len(s3_address_token_index_v2_1000))

Building V2 token indexes with threshold 1000...

S2 name index keys: 787893
S2 address index keys: 604579
S3 name index keys: 839333
S3 address index keys: 576658


In [ ]:
def generate_v2_block_candidates_1000(
    s1_row,
    top_k=2,
):
    """
    V2 blocker using:
      - exact normalized name
      - exact normalized address
      - 2 rarest name tokens with frequency <= 1000
      - 2 rarest address tokens with frequency <= 1000
    """

    candidates = set()

    normalized_name = s1_row["normalized_name_v2"]
    normalized_address = s1_row["normalized_address_v2"]

    # --------------------------------------------------------
    # Exact name
    # --------------------------------------------------------

    if normalized_name:
        candidates.update(
            s2_name_index_v2.get(normalized_name, [])
        )
        candidates.update(
            s3_name_index_v2.get(normalized_name, [])
        )

    # --------------------------------------------------------
    # Exact address
    # --------------------------------------------------------

    if normalized_address:
        candidates.update(
            s2_address_index_v2.get(normalized_address, [])
        )
        candidates.update(
            s3_address_index_v2.get(normalized_address, [])
        )

    # --------------------------------------------------------
    # Rare name tokens
    # --------------------------------------------------------

    if normalized_name:

        name_tokens = set(normalized_name.split())

        s2_tokens = [
            (token, s2_name_token_counts_v2[token])
            for token in name_tokens
            if (
                token in s2_name_token_counts_v2
                and s2_name_token_counts_v2[token] <= 1000
            )
        ]

        s2_tokens.sort(key=lambda x: x[1])

        for token, _ in s2_tokens[:top_k]:
            candidates.update(
                s2_name_token_index_v2_1000.get(token, [])
            )

        s3_tokens = [
            (token, s3_name_token_counts_v2[token])
            for token in name_tokens
            if (
                token in s3_name_token_counts_v2
                and s3_name_token_counts_v2[token] <= 1000
            )
        ]

        s3_tokens.sort(key=lambda x: x[1])

        for token, _ in s3_tokens[:top_k]:
            candidates.update(
                s3_name_token_index_v2_1000.get(token, [])
            )

    # --------------------------------------------------------
    # Rare address tokens
    # --------------------------------------------------------

    if normalized_address:

        address_tokens = set(normalized_address.split())

        s2_tokens = [
            (token, s2_address_token_counts_v2[token])
            for token in address_tokens
            if (
                token in s2_address_token_counts_v2
                and s2_address_token_counts_v2[token] <= 1000
            )
        ]

        s2_tokens.sort(key=lambda x: x[1])

        for token, _ in s2_tokens[:top_k]:
            candidates.update(
                s2_address_token_index_v2_1000.get(token, [])
            )

        s3_tokens = [
            (token, s3_address_token_counts_v2[token])
            for token in address_tokens
            if (
                token in s3_address_token_counts_v2
                and s3_address_token_counts_v2[token] <= 1000
            )
        ]

        s3_tokens.sort(key=lambda x: x[1])

        for token, _ in s3_tokens[:top_k]:
            candidates.update(
                s3_address_token_index_v2_1000.get(token, [])
            )

    return candidates

In [ ]:
test_row = validation_s1.iloc[0]

test_candidates_1000 = generate_v2_block_candidates_1000(
    test_row,
    top_k=2,
)

print("S1:", test_row["source1_entity_id"])
print("Threshold-1000 candidate count:", len(test_candidates_1000))
print(
    "First 20 candidates:",
    list(test_candidates_1000)[:20]
)

S1: S1-392794459
Threshold-1000 candidate count: 2298
First 20 candidates: ['S2-974357108', 'S3-648827143', 'S2-691641626', 'S2-573897508', 'S2-5778408', 'S3-657460581', 'S3-137458497', 'S3-540543859', 'S3-635839799', 'S2-239583320', 'S2-309817989', 'S3-92759088', 'S3-542169159', 'S2-338453075', 'S2-461219137', 'S3-999704781', 'S3-34980491', 'S2-277294125', 'S2-424190137', 'S2-293240316']


In [ ]:
# ============================================================
# V2 BLOCKER — THRESHOLD 1000, K=2
# ============================================================

start_time = time.time()

total_gt_pairs = 0
recovered_pairs = 0

s2_gt_pairs = 0
s2_recovered_pairs = 0

s3_gt_pairs = 0
s3_recovered_pairs = 0

complete_s1_matches = 0
zero_candidate_s1 = 0

candidate_counts_1000 = []


for row in validation_s1.itertuples(index=False):

    s1_id = row.source1_entity_id

    candidates = generate_v2_block_candidates_1000(
        row._asdict(),
        top_k=2,
    )

    candidate_count = len(candidates)
    candidate_counts_1000.append(candidate_count)

    if candidate_count == 0:
        zero_candidate_s1 += 1

    gt_ids = parse_gt_ids(
        valid_gt_lookup[s1_id]
    )

    total_gt_pairs += len(gt_ids)

    recovered = gt_ids.intersection(candidates)

    recovered_pairs += len(recovered)

    # --------------------------------------------------------
    # S2 / S3 breakdown
    # --------------------------------------------------------

    gt_s2 = {
        x for x in gt_ids
        if x.startswith("S2-")
    }

    gt_s3 = {
        x for x in gt_ids
        if x.startswith("S3-")
    }

    recovered_s2 = recovered.intersection(gt_s2)
    recovered_s3 = recovered.intersection(gt_s3)

    s2_gt_pairs += len(gt_s2)
    s2_recovered_pairs += len(recovered_s2)

    s3_gt_pairs += len(gt_s3)
    s3_recovered_pairs += len(recovered_s3)

    # --------------------------------------------------------
    # Complete S1
    # --------------------------------------------------------

    if gt_ids.issubset(candidates):
        complete_s1_matches += 1


candidate_counts_1000 = np.asarray(
    candidate_counts_1000
)

pair_recall_1000 = (
    recovered_pairs / total_gt_pairs
)

s2_recall_1000 = (
    s2_recovered_pairs / s2_gt_pairs
)

s3_recall_1000 = (
    s3_recovered_pairs / s3_gt_pairs
)

complete_s1_recall_1000 = (
    complete_s1_matches / len(validation_s1)
)

zero_candidate_rate_1000 = (
    zero_candidate_s1 / len(validation_s1)
)


print("=" * 70)
print("V2 BLOCKER — THRESHOLD 1000 / K=2")
print("=" * 70)

print(f"Validation S1 rows:       {len(validation_s1):,}")
print(f"Total GT pairs:           {total_gt_pairs:,}")
print(f"Recovered GT pairs:       {recovered_pairs:,}")
print()

print(f"Overall pair recall:      {pair_recall_1000:.6f}")
print(f"S2 pair recall:           {s2_recall_1000:.6f}")
print(f"S3 pair recall:           {s3_recall_1000:.6f}")
print()

print(
    f"Complete S1 recall:       "
    f"{complete_s1_recall_1000:.6f}"
)

print(f"Zero-candidate S1:        {zero_candidate_s1:,}")
print(
    f"Zero-candidate rate:      "
    f"{zero_candidate_rate_1000:.6f}"
)

print()

print(
    f"Mean candidates:          "
    f"{candidate_counts_1000.mean():.2f}"
)

print(
    f"Median candidates:        "
    f"{np.median(candidate_counts_1000):.0f}"
)

print(
    f"P90 candidates:           "
    f"{np.percentile(candidate_counts_1000, 90):.0f}"
)

print(
    f"P95 candidates:           "
    f"{np.percentile(candidate_counts_1000, 95):.0f}"
)

print(
    f"P99 candidates:           "
    f"{np.percentile(candidate_counts_1000, 99):.0f}"
)

print(
    f"Maximum candidates:       "
    f"{candidate_counts_1000.max():,}"
)

print()

print(
    f"Elapsed time:             "
    f"{time.time() - start_time:.2f} sec"
)

print("=" * 70)

V2 BLOCKER — THRESHOLD 1000 / K=2
Validation S1 rows:       441,365
Total GT pairs:           1,527,679
Recovered GT pairs:       1,387,355

Overall pair recall:      0.908146
S2 pair recall:           0.916161
S3 pair recall:           0.900630

Complete S1 recall:       0.808467
Zero-candidate S1:        2,273
Zero-candidate rate:      0.005150

Mean candidates:          1137.08
Median candidates:        910
P90 candidates:           2540
P95 candidates:           3076
P99 candidates:           4054
Maximum candidates:       7,576

Elapsed time:             82.69 sec


In [ ]:
# ============================================================
# V2 TOKEN INDEX — THRESHOLD 750
# ============================================================

V2_TOKEN_THRESHOLD_750 = 750

print("Building V2 token indexes with threshold 750...")

s2_name_token_index_v2_750 = build_token_index(
    s2["entity_id"],
    s2["normalized_name_v2"],
    s2_name_token_counts_v2,
    V2_TOKEN_THRESHOLD_750,
)

s2_address_token_index_v2_750 = build_token_index(
    s2["entity_id"],
    s2["normalized_address_v2"],
    s2_address_token_counts_v2,
    V2_TOKEN_THRESHOLD_750,
)

s3_name_token_index_v2_750 = build_token_index(
    s3["entity_id"],
    s3["normalized_name_v2"],
    s3_name_token_counts_v2,
    V2_TOKEN_THRESHOLD_750,
)

s3_address_token_index_v2_750 = build_token_index(
    s3["entity_id"],
    s3["normalized_address_v2"],
    s3_address_token_counts_v2,
    V2_TOKEN_THRESHOLD_750,
)

print()
print(
    "S2 name index keys:",
    len(s2_name_token_index_v2_750)
)

print(
    "S2 address index keys:",
    len(s2_address_token_index_v2_750)
)

print(
    "S3 name index keys:",
    len(s3_name_token_index_v2_750)
)

print(
    "S3 address index keys:",
    len(s3_address_token_index_v2_750)
)

Building V2 token indexes with threshold 750...

S2 name index keys: 787012
S2 address index keys: 603603
S3 name index keys: 838571
S3 address index keys: 575683


In [ ]:
# ============================================================
# V2 BLOCKER — THRESHOLD 750 / K=2
# ============================================================

def generate_v2_block_candidates_750(
    s1_row,
    top_k=2,
):
    candidates = set()

    normalized_name = s1_row["normalized_name_v2"]
    normalized_address = s1_row["normalized_address_v2"]

    # --------------------------------------------------------
    # Exact normalized name
    # --------------------------------------------------------

    if normalized_name:
        candidates.update(
            s2_name_index_v2.get(normalized_name, [])
        )
        candidates.update(
            s3_name_index_v2.get(normalized_name, [])
        )

    # --------------------------------------------------------
    # Exact normalized address
    # --------------------------------------------------------

    if normalized_address:
        candidates.update(
            s2_address_index_v2.get(normalized_address, [])
        )
        candidates.update(
            s3_address_index_v2.get(normalized_address, [])
        )

    # --------------------------------------------------------
    # Rare name tokens — threshold 750
    # --------------------------------------------------------

    if normalized_name:

        name_tokens = set(normalized_name.split())

        s2_tokens = [
            (token, s2_name_token_counts_v2[token])
            for token in name_tokens
            if (
                token in s2_name_token_counts_v2
                and s2_name_token_counts_v2[token] <= 750
            )
        ]

        s2_tokens.sort(key=lambda x: x[1])

        for token, _ in s2_tokens[:top_k]:
            candidates.update(
                s2_name_token_index_v2_750.get(token, [])
            )

        s3_tokens = [
            (token, s3_name_token_counts_v2[token])
            for token in name_tokens
            if (
                token in s3_name_token_counts_v2
                and s3_name_token_counts_v2[token] <= 750
            )
        ]

        s3_tokens.sort(key=lambda x: x[1])

        for token, _ in s3_tokens[:top_k]:
            candidates.update(
                s3_name_token_index_v2_750.get(token, [])
            )

    # --------------------------------------------------------
    # Rare address tokens — threshold 750
    # --------------------------------------------------------

    if normalized_address:

        address_tokens = set(normalized_address.split())

        s2_tokens = [
            (token, s2_address_token_counts_v2[token])
            for token in address_tokens
            if (
                token in s2_address_token_counts_v2
                and s2_address_token_counts_v2[token] <= 750
            )
        ]

        s2_tokens.sort(key=lambda x: x[1])

        for token, _ in s2_tokens[:top_k]:
            candidates.update(
                s2_address_token_index_v2_750.get(token, [])
            )

        s3_tokens = [
            (token, s3_address_token_counts_v2[token])
            for token in address_tokens
            if (
                token in s3_address_token_counts_v2
                and s3_address_token_counts_v2[token] <= 750
            )
        ]

        s3_tokens.sort(key=lambda x: x[1])

        for token, _ in s3_tokens[:top_k]:
            candidates.update(
                s3_address_token_index_v2_750.get(token, [])
            )

    return candidates


# ============================================================
# FULL VALIDATION
# ============================================================

start_time = time.time()

total_gt_pairs = 0
recovered_pairs = 0

s2_gt_pairs = 0
s2_recovered_pairs = 0

s3_gt_pairs = 0
s3_recovered_pairs = 0

complete_s1_matches = 0
zero_candidate_s1 = 0

candidate_counts_750 = []


for row in validation_s1.itertuples(index=False):

    s1_id = row.source1_entity_id

    candidates = generate_v2_block_candidates_750(
        row._asdict(),
        top_k=2,
    )

    candidate_count = len(candidates)
    candidate_counts_750.append(candidate_count)

    if candidate_count == 0:
        zero_candidate_s1 += 1

    gt_ids = parse_gt_ids(
        valid_gt_lookup[s1_id]
    )

    total_gt_pairs += len(gt_ids)

    recovered = gt_ids.intersection(candidates)

    recovered_pairs += len(recovered)

    # S2 / S3
    gt_s2 = {
        x for x in gt_ids
        if x.startswith("S2-")
    }

    gt_s3 = {
        x for x in gt_ids
        if x.startswith("S3-")
    }

    recovered_s2 = recovered.intersection(gt_s2)
    recovered_s3 = recovered.intersection(gt_s3)

    s2_gt_pairs += len(gt_s2)
    s2_recovered_pairs += len(recovered_s2)

    s3_gt_pairs += len(gt_s3)
    s3_recovered_pairs += len(recovered_s3)

    # Complete S1
    if gt_ids.issubset(candidates):
        complete_s1_matches += 1


candidate_counts_750 = np.asarray(candidate_counts_750)

pair_recall_750 = recovered_pairs / total_gt_pairs
s2_recall_750 = s2_recovered_pairs / s2_gt_pairs
s3_recall_750 = s3_recovered_pairs / s3_gt_pairs

complete_s1_recall_750 = (
    complete_s1_matches / len(validation_s1)
)

zero_candidate_rate_750 = (
    zero_candidate_s1 / len(validation_s1)
)


# ============================================================
# RESULTS
# ============================================================

print("=" * 70)
print("V2 BLOCKER — THRESHOLD 750 / K=2")
print("=" * 70)

print(f"Validation S1 rows:       {len(validation_s1):,}")
print(f"Total GT pairs:           {total_gt_pairs:,}")
print(f"Recovered GT pairs:       {recovered_pairs:,}")
print()

print(f"Overall pair recall:      {pair_recall_750:.6f}")
print(f"S2 pair recall:           {s2_recall_750:.6f}")
print(f"S3 pair recall:           {s3_recall_750:.6f}")
print()

print(
    f"Complete S1 recall:       "
    f"{complete_s1_recall_750:.6f}"
)

print(f"Zero-candidate S1:        {zero_candidate_s1:,}")
print(
    f"Zero-candidate rate:      "
    f"{zero_candidate_rate_750:.6f}"
)

print()

print(
    f"Mean candidates:          "
    f"{candidate_counts_750.mean():.2f}"
)

print(
    f"Median candidates:        "
    f"{np.median(candidate_counts_750):.0f}"
)

print(
    f"P90 candidates:           "
    f"{np.percentile(candidate_counts_750, 90):.0f}"
)

print(
    f"P95 candidates:           "
    f"{np.percentile(candidate_counts_750, 95):.0f}"
)

print(
    f"P99 candidates:           "
    f"{np.percentile(candidate_counts_750, 99):.0f}"
)

print(
    f"Maximum candidates:       "
    f"{candidate_counts_750.max():,}"
)

print()

print(
    f"Elapsed time:             "
    f"{time.time() - start_time:.2f} sec"
)

print("=" * 70)

V2 BLOCKER — THRESHOLD 750 / K=2
Validation S1 rows:       441,365
Total GT pairs:           1,527,679
Recovered GT pairs:       1,354,926

Overall pair recall:      0.886918
S2 pair recall:           0.895849
S3 pair recall:           0.878543

Complete S1 recall:       0.774114
Zero-candidate S1:        4,067
Zero-candidate rate:      0.009215

Mean candidates:          698.49
Median candidates:        565
P90 candidates:           1543
P95 candidates:           1877
P99 candidates:           2508
Maximum candidates:       4,825

Elapsed time:             62.77 sec


In [ ]:
# ============================================================
# MATCHER — STEP 1
# Generate candidate pairs for 10,000 validation S1 records
# ============================================================

MATCHER_SAMPLE_SIZE = 10_000

matcher_sample = validation_s1.sample(
    n=MATCHER_SAMPLE_SIZE,
    random_state=42,
).copy()

print("S1 sample:", len(matcher_sample))


# ------------------------------------------------------------
# Generate candidates
# ------------------------------------------------------------

candidate_pairs = []

start_time = time.time()

for row in matcher_sample.itertuples(index=False):

    s1_id = row.source1_entity_id

    candidates = generate_v2_block_candidates_1000(
        row._asdict(),
        top_k=2,
    )

    for target_id in candidates:
        candidate_pairs.append(
            {
                "source1_entity_id": s1_id,
                "target_entity_id": target_id,
            }
        )

print(
    f"Generated candidate pairs: {len(candidate_pairs):,}"
)

print(
    f"Elapsed: {time.time() - start_time:.2f} sec"
)

S1 sample: 10000
Generated candidate pairs: 11,445,411
Elapsed: 18.13 sec


In [ ]:
# ============================================================
# MATCHER — STEP 2
# Prepare candidate-pair dataframe
# ============================================================

candidate_pairs_df = pd.DataFrame(candidate_pairs)

print("Candidate pairs shape:", candidate_pairs_df.shape)
print()
print(candidate_pairs_df.head())
print()
print(candidate_pairs_df.dtypes)

Candidate pairs shape: (11445411, 2)

  source1_entity_id target_entity_id
0      S1-416888495     S3-173065903
1      S1-416888495     S2-556896272
2      S1-416888495     S3-434443855
3      S1-416888495     S3-146336080
4      S1-416888495      S3-38868558

source1_entity_id    str
target_entity_id     str
dtype: object


In [ ]:
# ============================================================
# Split S2 / S3 candidates
# ============================================================

candidate_pairs_df["target_source"] = (
    candidate_pairs_df["target_entity_id"]
    .str[:2]
)

print(
    candidate_pairs_df["target_source"]
    .value_counts()
)

target_source
S2    5794807
S3    5650604
Name: count, dtype: int64


In [ ]:
# ============================================================
# MATCHER — STEP 4
# Build S1 candidate dataframe
# ============================================================

s1_matcher = matcher_sample[
    [
        "source1_entity_id",
        "business_name",
        "business_address",
        "country",
        "normalized_name_v2",
        "normalized_address_v2",
    ]
].rename(
    columns={
        "business_name": "s1_business_name",
        "business_address": "s1_business_address",
        "country": "s1_country",
        "normalized_name_v2": "s1_name_v2",
        "normalized_address_v2": "s1_address_v2",
    }
)

candidate_features = candidate_pairs_df.merge(
    s1_matcher,
    on="source1_entity_id",
    how="left",
    validate="many_to_one",
)

print("After S1 join:", candidate_features.shape)

After S1 join: (11445411, 8)


In [ ]:
# ============================================================
# MATCHER — STEP 5
# Join target records from S2 and S3
# ============================================================

s2_matcher = s2[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "normalized_name_v2",
        "normalized_address_v2",
    ]
].rename(
    columns={
        "entity_id": "target_entity_id",
        "business_name": "target_business_name",
        "business_address": "target_business_address",
        "country": "target_country",
        "normalized_name_v2": "target_name_v2",
        "normalized_address_v2": "target_address_v2",
    }
)

s3_matcher = s3[
    [
        "entity_id",
        "business_name",
        "business_address",
        "country",
        "normalized_name_v2",
        "normalized_address_v2",
    ]
].rename(
    columns={
        "entity_id": "target_entity_id",
        "business_name": "target_business_name",
        "business_address": "target_business_address",
        "country": "target_country",
        "normalized_name_v2": "target_name_v2",
        "normalized_address_v2": "target_address_v2",
    }
)

# Split candidates by source
candidate_s2 = candidate_features[
    candidate_features["target_source"] == "S2"
].copy()

candidate_s3 = candidate_features[
    candidate_features["target_source"] == "S3"
].copy()

print("S2 candidate pairs:", len(candidate_s2))
print("S3 candidate pairs:", len(candidate_s3))

S2 candidate pairs: 5794807
S3 candidate pairs: 5650604


In [ ]:
candidate_s2 = candidate_s2.merge(
    s2_matcher,
    on="target_entity_id",
    how="left",
    validate="many_to_one",
)

candidate_s3 = candidate_s3.merge(
    s3_matcher,
    on="target_entity_id",
    how="left",
    validate="many_to_one",
)

candidate_features = pd.concat(
    [candidate_s2, candidate_s3],
    ignore_index=True,
)

print("Final candidate feature shape:", candidate_features.shape)
print()
print(
    candidate_features[
        [
            "s1_business_name",
            "target_business_name",
            "s1_country",
            "target_country",
        ]
    ].head()
)

Final candidate feature shape: (11445411, 13)

   s1_business_name      target_business_name s1_country target_country
0  Jai Tech Limited     रॉयल बिजनेस प्रा. लि.      India          India
1  Jai Tech Limited      जय टेक फूड्स लिमिटेड      India          India
2  Jai Tech Limited          Jai Tech Límited      India          India
3  Jai Tech Limited  Jai Tech  Bakery Limited      India          India
4  Jai Tech Limited    Royal Business Pvt Ltd      India          India


In [ ]:
# ============================================================
# MATCHER — STEP 6
# Cheap deterministic features
# ============================================================

candidate_features["country_same"] = (
    candidate_features["s1_country"]
    == candidate_features["target_country"]
).astype("int8")

candidate_features["name_exact"] = (
    candidate_features["s1_name_v2"]
    == candidate_features["target_name_v2"]
).astype("int8")

candidate_features["address_exact"] = (
    candidate_features["s1_address_v2"]
    == candidate_features["target_address_v2"]
).astype("int8")

candidate_features["name_length_s1"] = (
    candidate_features["s1_name_v2"].str.len()
)

candidate_features["name_length_target"] = (
    candidate_features["target_name_v2"].str.len()
)

candidate_features["address_length_s1"] = (
    candidate_features["s1_address_v2"].str.len()
)

candidate_features["address_length_target"] = (
    candidate_features["target_address_v2"].str.len()
)

candidate_features["name_length_diff"] = (
    candidate_features["name_length_s1"]
    - candidate_features["name_length_target"]
).abs()

candidate_features["address_length_diff"] = (
    candidate_features["address_length_s1"]
    - candidate_features["address_length_target"]
).abs()


print(
    candidate_features[
        [
            "country_same",
            "name_exact",
            "address_exact",
            "name_length_diff",
            "address_length_diff",
        ]
    ].describe()
)

       country_same    name_exact  address_exact  name_length_diff  \
count  1.144541e+07  1.144541e+07   1.144541e+07      1.144541e+07   
mean   9.203567e-01  9.315087e-03   4.534568e-04      8.732617e+00   
std    2.707402e-01  9.606413e-02   2.128970e-02      6.904423e+00   
min    0.000000e+00  0.000000e+00   0.000000e+00      0.000000e+00   
25%    1.000000e+00  0.000000e+00   0.000000e+00      3.000000e+00   
50%    1.000000e+00  0.000000e+00   0.000000e+00      7.000000e+00   
75%    1.000000e+00  0.000000e+00   0.000000e+00      1.300000e+01   
max    1.000000e+00  1.000000e+00   1.000000e+00      1.080000e+02   

       address_length_diff  
count         1.144541e+07  
mean          1.389234e+01  
std           1.518428e+01  
min           0.000000e+00  
25%           4.000000e+00  
50%           9.000000e+00  
75%           1.800000e+01  
max           1.880000e+02  


In [ ]:
print("Country same:")
print(candidate_features["country_same"].mean())

print("\nName exact:")
print(candidate_features["name_exact"].mean())

print("\nAddress exact:")
print(candidate_features["address_exact"].mean())

print("\nBoth name + address exact:")
print(
    (
        (candidate_features["name_exact"] == 1)
        &
        (candidate_features["address_exact"] == 1)
    ).mean()
)

Country same:
0.9203567263770607

Name exact:
0.0093150870685203

Address exact:
0.00045345684833860485

Both name + address exact:
8.370166872993901e-05


In [ ]:
# ============================================================
# MATCHER — STEP 7
# Token overlap features
# ============================================================

def token_jaccard_pair(a, b):
    if not a or not b:
        return 0.0

    a_tokens = set(a.split())
    b_tokens = set(b.split())

    union = a_tokens | b_tokens

    if not union:
        return 0.0

    return len(a_tokens & b_tokens) / len(union)


def token_containment_pair(a, b):
    if not a or not b:
        return 0.0

    a_tokens = set(a.split())
    b_tokens = set(b.split())

    if not a_tokens or not b_tokens:
        return 0.0

    return len(a_tokens & b_tokens) / min(
        len(a_tokens),
        len(b_tokens),
    )


start_time = time.time()

candidate_features["name_token_jaccard"] = [
    token_jaccard_pair(a, b)
    for a, b in zip(
        candidate_features["s1_name_v2"],
        candidate_features["target_name_v2"],
    )
]

candidate_features["name_token_containment"] = [
    token_containment_pair(a, b)
    for a, b in zip(
        candidate_features["s1_name_v2"],
        candidate_features["target_name_v2"],
    )
]

candidate_features["address_token_jaccard"] = [
    token_jaccard_pair(a, b)
    for a, b in zip(
        candidate_features["s1_address_v2"],
        candidate_features["target_address_v2"],
    )
]

candidate_features["address_token_containment"] = [
    token_containment_pair(a, b)
    for a, b in zip(
        candidate_features["s1_address_v2"],
        candidate_features["target_address_v2"],
    )
]

print(
    f"Elapsed: {time.time() - start_time:.2f} sec"
)

Elapsed: 175.18 sec


In [ ]:
print(
    candidate_features[
        [
            "name_token_jaccard",
            "name_token_containment",
            "address_token_jaccard",
            "address_token_containment",
        ]
    ].describe()
)

       name_token_jaccard  name_token_containment  address_token_jaccard  \
count        1.144541e+07            1.144541e+07           1.144541e+07   
mean         1.064395e-01            1.954238e-01           9.217108e-02   
std          1.524997e-01            2.371542e-01           1.010769e-01   
min          0.000000e+00            0.000000e+00           0.000000e+00   
25%          0.000000e+00            0.000000e+00           0.000000e+00   
50%          0.000000e+00            0.000000e+00           8.333333e-02   
75%          1.666667e-01            3.333333e-01           1.111111e-01   
max          1.000000e+00            1.000000e+00           1.000000e+00   

       address_token_containment  
count               1.144541e+07  
mean                1.779372e-01  
std                 1.649682e-01  
min                 0.000000e+00  
25%                 0.000000e+00  
50%                 1.666667e-01  
75%                 2.500000e-01  
max                 1.000000e+00  


In [ ]:
# ============================================================
# MATCHER — STEP 8
# Label candidate pairs using validation ground truth
# ============================================================

# Build a set of true pairs for the 10k validation sample
positive_pairs = set()

for s1_id, gt_value in valid_gt_lookup.items():

    # Only include our 10k matcher sample
    if s1_id not in set(matcher_sample["source1_entity_id"]):
        continue

    gt_ids = parse_gt_ids(gt_value)

    for target_id in gt_ids:
        positive_pairs.add(
            (s1_id, target_id)
        )

print("True pairs in matcher sample:", len(positive_pairs))


# ------------------------------------------------------------
# Label candidates
# ------------------------------------------------------------

candidate_features["is_match"] = [
    (s1_id, target_id) in positive_pairs
    for s1_id, target_id in zip(
        candidate_features["source1_entity_id"],
        candidate_features["target_entity_id"],
    )
]

candidate_features["is_match"] = (
    candidate_features["is_match"].astype("int8")
)

print()
print(
    candidate_features["is_match"].value_counts()
)

print()
print(
    candidate_features["is_match"].mean()
)

True pairs in matcher sample: 34714

is_match
0    11413760
1       31651
Name: count, dtype: int64

0.0027653878047717117


In [ ]:
# ============================================================
# Compare feature distributions: positives vs negatives
# ============================================================

feature_cols = [
    "country_same",
    "name_exact",
    "address_exact",
    "name_length_diff",
    "address_length_diff",
    "name_token_jaccard",
    "name_token_containment",
    "address_token_jaccard",
    "address_token_containment",
]

comparison = (
    candidate_features
    .groupby("is_match")[feature_cols]
    .mean()
    .T
)

comparison.columns = [
    "negative",
    "positive",
]

comparison["difference"] = (
    comparison["positive"]
    - comparison["negative"]
)

print(comparison)

                            negative  positive  difference
country_same                0.920136  1.000000    0.079864
name_exact                  0.008552  0.284320    0.275767
address_exact               0.000084  0.133645    0.133561
name_length_diff            8.747280  3.444694   -5.302587
address_length_diff        13.907423  8.454899   -5.452524
name_token_jaccard          0.104881  0.668553    0.563672
name_token_containment      0.193747  0.800238    0.606492
address_token_jaccard       0.090547  0.677738    0.587191
address_token_containment   0.176143  0.825035    0.648892


In [ ]:
import importlib.util

print("LightGBM installed:",
      importlib.util.find_spec("lightgbm") is not None)

LightGBM installed: True


In [ ]:
from lightgbm import LGBMClassifier

feature_cols = [
    "country_same",
    "name_exact",
    "address_exact",
    "name_length_diff",
    "address_length_diff",
    "name_token_jaccard",
    "name_token_containment",
    "address_token_jaccard",
    "address_token_containment",
]

X = candidate_features[feature_cols]
y = candidate_features["is_match"]

model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(X, y)

print("Model trained.")

[LightGBM] [Info] Number of positive: 31651, number of negative: 11413760
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.538490 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 539
[LightGBM] [Info] Number of data points in the train set: 11445411, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Model trained.


In [ ]:
candidate_features["match_probability"] = model.predict_proba(
    candidate_features[feature_cols]
)[:, 1]

print(candidate_features["match_probability"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
))

count    1.144541e+07
mean     1.743663e-02
std      1.015142e-01
min      9.816221e-07
50%      9.936460e-05
90%      5.431827e-03
95%      3.571370e-02
99%      6.632167e-01
99.9%    9.998451e-01
max      1.000000e+00
Name: match_probability, dtype: float64


In [ ]:
from sklearn.metrics import precision_score, recall_score, fbeta_score

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

y_true = candidate_features["is_match"].values
probs = candidate_features["match_probability"].values

results = []

for threshold in thresholds:
    y_pred = (probs >= threshold).astype("int8")

    results.append({
        "threshold": threshold,
        "predicted_matches": y_pred.sum(),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f0.5": fbeta_score(y_true, y_pred, beta=0.5, zero_division=0),
    })

threshold_results = pd.DataFrame(results)

print(threshold_results)

    threshold  predicted_matches  precision    recall      f0.5
0        0.10             368547   0.085859  0.999747  0.105068
1        0.20             266633   0.118590  0.999021  0.143965
2        0.30             212139   0.148808  0.997378  0.179321
3        0.40             175587   0.179393  0.995198  0.214571
4        0.50             146184   0.214962  0.992828  0.254905
5        0.60             125931   0.248803  0.989921  0.292617
6        0.70             106757   0.292112  0.985277  0.339944
7        0.80              87478   0.354066  0.978579  0.405870
8        0.90              68358   0.447775  0.967078  0.501650
9        0.95              55789   0.539676  0.951250  0.590800
10       0.99              36929   0.756127  0.882215  0.778376


In [ ]:
fine_thresholds = [
    0.990, 0.992, 0.994, 0.996,
    0.998, 0.999, 0.9995, 0.9999
]

results = []

for threshold in fine_thresholds:
    y_pred = (probs >= threshold).astype("int8")

    results.append({
        "threshold": threshold,
        "predicted_matches": int(y_pred.sum()),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f0.5": fbeta_score(y_true, y_pred, beta=0.5, zero_division=0),
    })

fine_results = pd.DataFrame(results)

print(fine_results)

   threshold  predicted_matches  precision    recall      f0.5
0     0.9900              36929   0.756127  0.882215  0.778376
1     0.9920              35480   0.776268  0.870178  0.793393
2     0.9940              32879   0.813346  0.844902  0.819467
3     0.9960              29593   0.860406  0.804461  0.848603
4     0.9980              24551   0.921958  0.715143  0.871549
5     0.9990              20358   0.963945  0.620012  0.867681
6     0.9995              17118   0.983000  0.531642  0.840316
7     0.9999               9261   0.993413  0.290670  0.669627


In [ ]:
threshold = 0.998

predicted_pairs = candidate_features[
    candidate_features["match_probability"] >= threshold
]

prediction_counts = (
    predicted_pairs
    .groupby("source1_entity_id")
    .size()
)

print(prediction_counts.describe())
print("\nCount distribution:")
print(prediction_counts.value_counts().sort_index().head(20))

count    8840.000000
mean        2.777262
std         1.435589
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        10.000000
dtype: float64

Count distribution:
1     1884
2     2299
3     2188
4     1384
5      682
6      300
7       73
8       23
9        6
10       1
Name: count, dtype: int64


In [ ]:
# Attach ground-truth match count to each S1
gt_count_map = {
    s1_id: len(parse_gt_ids(gt_value))
    for s1_id, gt_value in valid_gt_lookup.items()
}

count_check = prediction_counts.rename("predicted_count").to_frame()

count_check["true_count"] = (
    count_check.index.map(gt_count_map).fillna(0).astype(int)
)

count_check["count_diff"] = (
    count_check["predicted_count"] - count_check["true_count"]
)

print(count_check.describe())

print("\nPrediction count vs true count:")
print(
    count_check[
        ["predicted_count", "true_count"]
    ].value_counts().sort_index()
)

       predicted_count   true_count   count_diff
count      8840.000000  8840.000000  8840.000000
mean          2.777262     3.716290    -0.939027
std           1.435589     1.567231     1.235489
min           1.000000     0.000000    -7.000000
25%           2.000000     3.000000    -2.000000
50%           3.000000     4.000000    -1.000000
75%           4.000000     5.000000     0.000000
max          10.000000    10.000000     5.000000

Prediction count vs true count:
predicted_count  true_count
1                0             118
                 1             305
                 2             568
                 3             485
                 4             264
                              ... 
9                5               1
                 7               1
                 8               2
                 9               2
10               9               1
Name: count, Length: 67, dtype: int64


In [ ]:
# Build predicted sets at threshold 0.998
threshold = 0.998

predicted_sets = (
    candidate_features[
        candidate_features["match_probability"] >= threshold
    ]
    .groupby("source1_entity_id")["target_entity_id"]
    .apply(set)
)

# Evaluate set-level precision/recall
set_results = []

for s1_id, gt_value in valid_gt_lookup.items():
    if s1_id not in set(predicted_sets.index):
        pred = set()
    else:
        pred = predicted_sets[s1_id]

    gt = set(parse_gt_ids(gt_value))

    tp = len(pred & gt)
    fp = len(pred - gt)
    fn = len(gt - pred)

    set_results.append((tp, fp, fn))

tp = sum(x[0] for x in set_results)
fp = sum(x[1] for x in set_results)
fn = sum(x[2] for x in set_results)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f05 = (1.25 * precision * recall) / (0.25 * precision + recall) if precision + recall else 0

print("Set-level TP:", tp)
print("Set-level FP:", fp)
print("Set-level FN:", fn)
print("Precision:", precision)
print("Recall:", recall)
print("F0.5:", f05)

Set-level TP: 22635
Set-level FP: 1916
Set-level FN: 1505044
Precision: 0.9219583723677244
Recall: 0.014816594323807554
F0.5: 0.06960832975066472


In [ ]:
# Build predicted sets at threshold 0.99
threshold = 0.99

predicted_sets = (
    candidate_features[
        candidate_features["match_probability"] >= threshold
    ]
    .groupby("source1_entity_id")["target_entity_id"]
    .apply(set)
)

# Evaluate set-level precision/recall
set_results = []

for s1_id, gt_value in valid_gt_lookup.items():
    if s1_id not in set(predicted_sets.index):
        pred = set()
    else:
        pred = predicted_sets[s1_id]

    gt = set(parse_gt_ids(gt_value))

    tp = len(pred & gt)
    fp = len(pred - gt)
    fn = len(gt - pred)

    set_results.append((tp, fp, fn))

tp = sum(x[0] for x in set_results)
fp = sum(x[1] for x in set_results)
fn = sum(x[2] for x in set_results)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
f05 = (1.25 * precision * recall) / (0.25 * precision + recall) if precision + recall else 0

print("Set-level TP:", tp)
print("Set-level FP:", fp)
print("Set-level FN:", fn)
print("Precision:", precision)
print("Recall:", recall)
print("F0.5:", f05)

Set-level TP: 27923
Set-level FP: 9006
Set-level FN: 1499756
Precision: 0.756126621354491
Recall: 0.01827805448657735
F0.5: 0.08333258724062087


In [ ]:
# True pairs that actually exist inside our candidate set
candidate_positive_count = candidate_features["is_match"].sum()

# Total true pairs for these 10k S1s
total_true_pairs = len(positive_pairs)

print("Total true pairs:", total_true_pairs)
print("True pairs inside candidates:", candidate_positive_count)
print(
    "Candidate recall:",
    candidate_positive_count / total_true_pairs
)

Total true pairs: 34714
True pairs inside candidates: 31651
Candidate recall: 0.9117647058823529


In [ ]:
# Matcher performance at threshold 0.99,
# restricted ONLY to true pairs that survived blocking

threshold = 0.99

blocked_true_pairs = candidate_features[
    candidate_features["is_match"] == 1
]

matcher_recovered = (
    blocked_true_pairs["match_probability"] >= threshold
).sum()

matcher_recall = matcher_recovered / len(blocked_true_pairs)

print("Blocked true pairs:", len(blocked_true_pairs))
print("Recovered by matcher:", matcher_recovered)
print("Matcher recall:", matcher_recall)

print(
    "Overall pair recall:",
    matcher_recovered / total_true_pairs
)

Blocked true pairs: 31651
Recovered by matcher: 27923
Matcher recall: 0.882215411835329
Overall pair recall: 0.8043728754969177


In [ ]:
false_negatives = candidate_features[
    (candidate_features["is_match"] == 1) &
    (candidate_features["match_probability"] < 0.99)
].copy()

print("False negatives:", len(false_negatives))

print(
    false_negatives[
        [
            "source1_entity_id",
            "target_entity_id",
            "match_probability",
            "country_same",
            "name_exact",
            "address_exact",
            "name_token_jaccard",
            "name_token_containment",
            "address_token_jaccard",
            "address_token_containment",
        ]
    ]
    .sort_values("match_probability")
    .head(20)
)


False negatives: 3728
         source1_entity_id target_entity_id  match_probability  country_same  \
7072503        S1-32646687     S3-691142472           0.011948             1   
10541         S1-974472894     S2-985873022           0.017476             1   
10538664       S1-25648499     S3-693515259           0.040972             1   
10564263      S1-889328499     S3-814629167           0.042077             1   
7587458       S1-245295831     S3-607523186           0.071131             1   
2307802       S1-438209565       S2-7567980           0.083964             1   
10538663       S1-25648499     S3-233431501           0.093713             1   
5737823       S1-502245239     S2-696974502           0.093743             1   
8510073       S1-144947812      S3-21201690           0.109309             1   
8996156       S1-261184770     S3-571663384           0.111475             1   
8617875       S1-202334616      S3-42969031           0.117420             1   
8383946       S1-2

In [ ]:
import importlib.util

print("RapidFuzz installed:",
      importlib.util.find_spec("rapidfuzz") is not None)

RapidFuzz installed: True


In [ ]:
from rapidfuzz.fuzz import ratio, WRatio

candidate_features["name_char_ratio"] = [
    ratio(a, b) / 100
    for a, b in zip(
        candidate_features["s1_name_v2"],
        candidate_features["target_name_v2"]
    )
]

candidate_features["address_char_ratio"] = [
    ratio(a, b) / 100
    for a, b in zip(
        candidate_features["s1_address_v2"],
        candidate_features["target_address_v2"]
    )
]

candidate_features["name_wratio"] = [
    WRatio(a, b) / 100
    for a, b in zip(
        candidate_features["s1_name_v2"],
        candidate_features["target_name_v2"]
    )
]

candidate_features["address_wratio"] = [
    WRatio(a, b) / 100
    for a, b in zip(
        candidate_features["s1_address_v2"],
        candidate_features["target_address_v2"]
    )
]

print(
    candidate_features[
        [
            "name_char_ratio",
            "address_char_ratio",
            "name_wratio",
            "address_wratio"
        ]
    ].describe()
)

       name_char_ratio  address_char_ratio   name_wratio  address_wratio
count     1.144541e+07        1.144541e+07  1.144541e+07    1.144541e+07
mean      4.096067e-01        4.117148e-01  5.101838e-01    5.199865e-01
std       1.577433e-01        1.196355e-01  2.132014e-01    1.922121e-01
min       0.000000e+00        0.000000e+00  0.000000e+00    0.000000e+00
25%       3.076923e-01        3.389831e-01  3.562500e-01    3.848101e-01
50%       3.859649e-01        3.953488e-01  4.460870e-01    4.642857e-01
75%       5.000000e-01        4.731183e-01  6.551724e-01    6.175000e-01
max       1.000000e+00        1.000000e+00  1.000000e+00    1.000000e+00


In [ ]:
new_feature_cols = [
    "name_char_ratio",
    "address_char_ratio",
    "name_wratio",
    "address_wratio",
]

comparison_new = (
    candidate_features
    .groupby("is_match")[new_feature_cols]
    .mean()
    .T
)

comparison_new.columns = ["negative", "positive"]
comparison_new["difference"] = (
    comparison_new["positive"] - comparison_new["negative"]
)

print(comparison_new)

                    negative  positive  difference
name_char_ratio     0.408478  0.816645    0.408167
address_char_ratio  0.410666  0.789915    0.379249
name_wratio         0.509167  0.876686    0.367519
address_wratio      0.518993  0.878343    0.359350


In [ ]:
feature_cols_v2 = [
    "country_same",
    "name_exact",
    "address_exact",
    "name_length_diff",
    "address_length_diff",
    "name_token_jaccard",
    "name_token_containment",
    "address_token_jaccard",
    "address_token_containment",
    "name_char_ratio",
    "address_char_ratio",
    "name_wratio",
    "address_wratio",
]

X_v2 = candidate_features[feature_cols_v2]
y = candidate_features["is_match"]

model_v2 = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model_v2.fit(X_v2, y)

print("Model v2 trained.")

[LightGBM] [Info] Number of positive: 31651, number of negative: 11413760
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.082808 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1559
[LightGBM] [Info] Number of data points in the train set: 11445411, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Model v2 trained.


In [ ]:
candidate_features["match_probability_v2"] = model_v2.predict_proba(
    candidate_features[feature_cols_v2]
)[:, 1]

print(
    candidate_features["match_probability_v2"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
    )
)

count    1.144541e+07
mean     8.173104e-03
std      7.647085e-02
min      6.353623e-13
50%      4.975286e-05
90%      8.787074e-04
95%      3.649938e-03
99%      1.778637e-01
99.9%    9.998640e-01
max      9.999850e-01
Name: match_probability_v2, dtype: float64


In [ ]:
thresholds_v2 = [
    0.90, 0.95, 0.98, 0.99, 0.992,
    0.994, 0.996, 0.998, 0.999
]

y_true = candidate_features["is_match"].values
probs_v2 = candidate_features["match_probability_v2"].values

results_v2 = []

for threshold in thresholds_v2:
    y_pred = (probs_v2 >= threshold).astype("int8")

    results_v2.append({
        "threshold": threshold,
        "predicted_matches": int(y_pred.sum()),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f0.5": fbeta_score(y_true, y_pred, beta=0.5, zero_division=0),
    })

threshold_results_v2 = pd.DataFrame(results_v2)

print(threshold_results_v2)

   threshold  predicted_matches  precision    recall      f0.5
0      0.900              50508   0.618793  0.987457  0.668726
1      0.950              45926   0.674324  0.978452  0.719022
2      0.980              41749   0.730604  0.963698  0.767744
3      0.990              35718   0.821435  0.926985  0.840577
4      0.992              34607   0.837894  0.916148  0.852457
5      0.994              33026   0.859293  0.896623  0.866508
6      0.996              31290   0.882071  0.872010  0.880040
7      0.998              27497   0.930283  0.808189  0.903000
8      0.999              22362   0.973079  0.687498  0.898438


In [ ]:
threshold = 0.998

predicted_sets_v2 = (
    candidate_features[
        candidate_features["match_probability_v2"] >= threshold
    ]
    .groupby("source1_entity_id")["target_entity_id"]
    .apply(set)
)

set_results_v2 = []

for s1_id, gt_value in valid_gt_lookup.items():
    pred = predicted_sets_v2.get(s1_id, set())
    gt = set(parse_gt_ids(gt_value))

    tp = len(pred & gt)
    fp = len(pred - gt)
    fn = len(gt - pred)

    set_results_v2.append((tp, fp, fn))

tp = sum(x[0] for x in set_results_v2)
fp = sum(x[1] for x in set_results_v2)
fn = sum(x[2] for x in set_results_v2)

precision_v2 = tp / (tp + fp) if tp + fp else 0
recall_v2 = tp / (tp + fn) if tp + fn else 0
f05_v2 = (
    1.25 * precision_v2 * recall_v2 /
    (0.25 * precision_v2 + recall_v2)
    if precision_v2 + recall_v2 else 0
)

print("Set-level TP:", tp)
print("Set-level FP:", fp)
print("Set-level FN:", fn)
print("Precision:", precision_v2)
print("Recall:", recall_v2)
print("F0.5:", f05_v2)

Set-level TP: 25580
Set-level FP: 1917
Set-level FN: 1502099
Precision: 0.9302833036331236
Recall: 0.016744355325955255
F0.5: 0.07809890533301336


In [ ]:
# Correct set-level evaluation: ONLY the 10k matcher sample

sample_s1_ids = set(matcher_sample["source1_entity_id"])

set_results_v2 = []

for s1_id in sample_s1_ids:
    gt_value = valid_gt_lookup[s1_id]

    pred = predicted_sets_v2.get(s1_id, set())
    gt = set(parse_gt_ids(gt_value))

    tp = len(pred & gt)
    fp = len(pred - gt)
    fn = len(gt - pred)

    set_results_v2.append((tp, fp, fn))

tp = sum(x[0] for x in set_results_v2)
fp = sum(x[1] for x in set_results_v2)
fn = sum(x[2] for x in set_results_v2)

precision_v2 = tp / (tp + fp) if tp + fp else 0
recall_v2 = tp / (tp + fn) if tp + fn else 0
f05_v2 = (
    1.25 * precision_v2 * recall_v2 /
    (0.25 * precision_v2 + recall_v2)
    if precision_v2 + recall_v2 else 0
)

print("Set-level TP:", tp)
print("Set-level FP:", fp)
print("Set-level FN:", fn)
print("Precision:", precision_v2)
print("Recall:", recall_v2)
print("F0.5:", f05_v2)

Set-level TP: 25580
Set-level FP: 1917
Set-level FN: 9134
Precision: 0.9302833036331236
Recall: 0.7368784928271015
F0.5: 0.8838855026191759


In [ ]:
set_thresholds = [0.990, 0.992, 0.994, 0.996, 0.998, 0.999]

set_eval = []

for threshold in set_thresholds:

    predicted_sets = (
        candidate_features[
            candidate_features["match_probability_v2"] >= threshold
        ]
        .groupby("source1_entity_id")["target_entity_id"]
        .apply(set)
    )

    tp = fp = fn = 0

    for s1_id in sample_s1_ids:
        gt = set(parse_gt_ids(valid_gt_lookup[s1_id]))
        pred = predicted_sets.get(s1_id, set())

        tp += len(pred & gt)
        fp += len(pred - gt)
        fn += len(gt - pred)

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f05 = (
        1.25 * precision * recall /
        (0.25 * precision + recall)
        if precision + recall else 0
    )

    set_eval.append({
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "F0.5": f05,
    })

set_eval_df = pd.DataFrame(set_eval)

print(set_eval_df)

   threshold     TP    FP     FN  precision    recall      F0.5
0      0.990  29340  6378   5374   0.821435  0.845192  0.826079
1      0.992  28997  5610   5717   0.837894  0.835311  0.837376
2      0.994  28379  4647   6335   0.859293  0.817509  0.850598
3      0.996  27600  3690   7114   0.882071  0.795068  0.863180
4      0.998  25580  1917   9134   0.930283  0.736878  0.883886
5      0.999  21760   602  12954   0.973079  0.626836  0.876275


In [ ]:
blocking_misses = candidate_features[
    candidate_features["is_match"] == 0
].copy()

# This is only the candidate dataframe, so instead recover
# all true pairs that are absent from candidates.

candidate_pair_set = set(
    zip(
        candidate_features["source1_entity_id"],
        candidate_features["target_entity_id"]
    )
)

missed_true_pairs = []

for s1_id in sample_s1_ids:
    gt_ids = parse_gt_ids(valid_gt_lookup[s1_id])

    for target_id in gt_ids:
        if (s1_id, target_id) not in candidate_pair_set:
            missed_true_pairs.append((s1_id, target_id))

print("Missed true pairs:", len(missed_true_pairs))
print("Blocking recall:", 1 - len(missed_true_pairs) / total_true_pairs)